In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
pCO2 training script with convenient modeling-scope switch
Default target: direct pCO2 regression
FINAL_TARGET in {"r_star_T", "pCO2"}

---------------------------------------------------------
Supported targets:
1) FINAL_TARGET = "pCO2"
   Direct pCO2 regression:
       pCO2_pred = model(X)

2) FINAL_TARGET = "r_star_T"
   Thermal-removed target relative to the thermal prior:
       thermal_prior = pCO2_prior_clim + delta_pCO2_T
       r_star_T = pCO2 - thermal_prior
       pCO2_pred = thermal_prior + r_star_T_pred

---------------------------------------------------------
New modeling-scope switch (recommended for reproducibility):
- MODELING_MODE = "zonewise"
    Train one model per Zone (uses ZONE_COL)

- MODELING_MODE = "global"
    Train one single global model on all samples
    Completely ignores ZONE_COL during model fitting

Important principle:
- The framework is identical between the two modes:
    feature engineering
    feature screening / selection
    inverse-density weighting
    Optuna tuning
    ensemble CatBoost
    DecadeBlockKFold validation
    prior-only baseline evaluation
- Only the training scope changes.
This is the most审稿友好 way to compare "zone-wise" vs "global unified" modeling.

---------------------------------------------------------
"""

import os
import gc
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFECV

import joblib

from catboost import CatBoostRegressor, Pool
import optuna
from optuna.samplers import TPESampler
from math import sqrt


# =========================================================
# 0. Global config
# =========================================================
ENABLE_PLOTTING = True

# ---------------------------------------------------------
# Run mode
# ---------------------------------------------------------
RUN_MODE = "main_plus_prior_baseline"
VALID_RUN_MODES = {"main_only", "main_plus_prior_baseline"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"Invalid RUN_MODE={RUN_MODE}. Must be one of {sorted(VALID_RUN_MODES)}")

ENABLE_PRIOR_BASELINE = (RUN_MODE == "main_plus_prior_baseline")

# ---------------------------------------------------------
# Modeling mode (NEW)
# ---------------------------------------------------------
MODELING_MODE = "zonewise"   # {"zonewise", "global"}
VALID_MODELING_MODES = {"zonewise", "global"}
if MODELING_MODE not in VALID_MODELING_MODES:
    raise ValueError(f"Invalid MODELING_MODE={MODELING_MODE}. Must be one of {sorted(VALID_MODELING_MODES)}")

ENABLE_ZONE_MODELING = (MODELING_MODE == "zonewise")

# ---------------------------------------------------------
# Ensemble config
# ---------------------------------------------------------
ENABLE_ENSEMBLE = True
ENSEMBLE_SIZE = 20
ENSEMBLE_BASE_SEED = 4200
ENSEMBLE_BOOTSTRAP_ROWS = True
ENSEMBLE_BOOTSTRAP_FRAC = 1.0
SAVE_MEMBER_MODELS = True

# ---------------------------------------------------------
# Feature engineering / selection config
# ---------------------------------------------------------
ENABLE_FEATURE_ENGINEERING = True
ENABLE_FEATURE_SELECTION = True

PERM_N_SPLITS = 5
PERM_N_REPEATS = 3
PERM_TOPK = 12
PERM_MAX_VAL_SAMPLES = 12000

RFECV_STEP = 1
RFECV_SCORING = "neg_root_mean_squared_error"
USE_ADAPTIVE_RFECV_MIN = True
RFECV_MIN_RATIO = 0.50
RFECV_MIN_ABS = 8
RFECV_MIN_FEATS = 8

# ---- Inverse-density weighting ----
ENABLE_DENSITY_WEIGHTING = True
WEIGHT_GRID_RESOLUTION_DEG = 5.0
WEIGHT_TIME_RESOLUTION_YR = 10.0

ENABLE_TRANSITION_PERIOD_PENALTY = False
PENALTY_YEAR_START = 1982
PENALTY_YEAR_END = 2000
PENALTY_FACTOR = 0.5

# ---- Optuna ----
N_TRIALS_OPTUNA = int(os.environ.get("N_TRIALS_OPTUNA", "36"))
RANDOM_SEED = 42

MAX_CORES = 24
TOTAL_CORES = min(os.cpu_count() or 16, MAX_CORES)
_DEFAULT_CB_THREADS = max(4, TOTAL_CORES)
N_THREADS = int(os.environ.get("CB_N_THREADS", str(_DEFAULT_CB_THREADS)))

# Avoid BLAS/OMP oversubscription
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"


# =========================================================
# 1. Data / split config
# =========================================================
CSV_PATH = "/data/wang/Result_pCO2/allpco2/SOCAT_TRAIN.csv"

# Use root + mode subdirectories to avoid overwriting zonewise and global results.
OUTPUT_ROOT_DIR = "/data/wang/Result_pCO2/models_ML/current_model"
OUTPUT_DIR = os.path.join(
    OUTPUT_ROOT_DIR,
    "zonewise_model" if ENABLE_ZONE_MODELING else "global_model"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

ZONE_COL = "Zone"

YEAR_START = 1982
YEAR_END = 2024

TEST_YEARS = [1984, 1988, 1995, 2004, 2013, 2021]
MIN_SAMPLES = 300


# =========================================================
# 2. Field candidates
# =========================================================
DATE_CANDIDATES = ["Date"]
LABEL_CANDIDATES = ["pCO2"]
PRIOR_CANDIDATES = ["pCO2_prior"]
PRIOR_CLIM_CANDIDATES = ["pCO2_prior_clim_mapped", "pco2_prior_clim_mapped", "pCO2_prior_clim", "pco2_prior_clim"]
R_STAR_T_CANDIDATES = ["r_star_T"]
DELTA_PCO2_T_CANDIDATES = ["delta_pCO2_T"]

TIME_COL = "Year"
MONTH_COL = "Month"
LAT_COL = "Latitude"
LON_COL = "Longitude"

VALID_FINAL_TARGETS = {"r_star_T", "pCO2"}
# Default setting: direct pCO2 regression
FINAL_TARGET = "r_star_T"
if FINAL_TARGET not in VALID_FINAL_TARGETS:
    raise ValueError(f"Invalid FINAL_TARGET={FINAL_TARGET}. Must be one of {sorted(VALID_FINAL_TARGETS)}")

FINAL_PRIOR_NAME = "pCO2_prior_used"
FINAL_PRIOR_CLIM_NAME = "pCO2_prior_clim_used"
FINAL_LABEL_NAME = "pCO2_label_used"
FINAL_DELTA_T_NAME = "delta_pCO2_T_used"
FINAL_RECON_OFFSET_NAME = "pCO2_reconstruction_offset_used"
FINAL_BASELINE_PCO2_NAME = "pCO2_baseline_used"

# ---------------------------------------------------------
# Compact feature design
# ---------------------------------------------------------
BASE_ALWAYS_KEEP = []

CORE_FEATURES = [
    "geo_nz",
    "CO2",
    "SST_anom",
    "SSS_anom",
    "SST",
    "SSS",
    "log10_Chla",
    "SLA_ERA5",
    "NINO34",
    "Siconc",
    "MLD",
    "U",
    "V",
    "pH","DIC",
    'pH_anom', 'DIC_anom',
]

INTERACTION_FEATURES = [
    #"SSTc_x_DOc",
    #"SSTc_x_SSSc",
    #"DOc_x_log10Chla",
]

ALL_CANDIDATE_FEATURES = list(dict.fromkeys(BASE_ALWAYS_KEEP + CORE_FEATURES + INTERACTION_FEATURES))
ALWAYS_KEEP = list(BASE_ALWAYS_KEEP)

INCLUDE_PRIOR_AS_PREDICTOR = False
PRIOR_AS_FEATURE_CANDIDATES = ["pCO2_prior", "pco2_prior_clim_mapped"]

ENFORCE_VALID_FEATURES = True

SENTINEL_VALUES = [
    -9999, -9999.0, -999, -32767, 32767,
    1e20, -1e20, 9.96921e36, -9.96921e36
]

FEATURE_RANGES = {
    "Latitude": (-90, 90),
    "Longitude": (-180, 360),
    "Month": (1, 12),
    "Year": (1800, 2100),
    "pCO2": (0, 1100),
}


# =========================================================
# 3. Plot style
# =========================================================
if ENABLE_PLOTTING:
    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "axes.linewidth": 1.1,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "savefig.dpi": 340,
        "figure.figsize": (6.4, 4.8),
    })


# =========================================================
# 4. General helpers
# =========================================================
def find_first_existing(columns, candidates, required=False, field_desc="field"):
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise KeyError(f"Missing required {field_desc}. Candidates: {candidates}")
    return None


def compute_chla_offset(x):
    x = np.asarray(x, float)
    pos = x[np.isfinite(x) & (x > 0)]
    if pos.size == 0:
        return 1e-6
    p5 = np.nanpercentile(pos, 5)
    return max(1e-6, 0.01 * p5)


def get_log10_chla(df, chla_offset=None):
    if "log10_Chla" in df.columns:
        return pd.to_numeric(df["log10_Chla"], errors="coerce")
    if "Chla" not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)
    z = pd.to_numeric(df["Chla"], errors="coerce").to_numpy(dtype=float, copy=False)
    if chla_offset is None:
        chla_offset = compute_chla_offset(z)
    out = np.where(np.isfinite(z), np.log10(np.maximum(z, 0.0) + float(chla_offset)), np.nan)
    return pd.Series(out, index=df.index, dtype=float)


def calibration_stats(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size == 0:
        return {"slope": np.nan, "intercept": np.nan, "bias": np.nan, "r2": np.nan}
    try:
        lr = LinearRegression().fit(y_true.reshape(-1, 1), y_pred)
        return dict(
            slope=float(lr.coef_[0]),
            intercept=float(lr.intercept_),
            bias=float(np.nanmean(y_pred - y_true)),
            r2=float(r2_score(y_true, y_pred))
        )
    except Exception:
        return {"slope": np.nan, "intercept": np.nan, "bias": np.nan, "r2": np.nan}


def safe_mape(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    m = np.isfinite(y_true) & np.isfinite(y_pred) & (np.abs(y_true) > eps)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs((y_pred[m] - y_true[m]) / y_true[m])) * 100.0)


def compute_metrics_dict(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if y_true.size == 0:
        return {"MSE": np.nan, "RMSE": np.nan, "MAE": np.nan, "R2": np.nan, "MAPE": np.nan}
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    mape = safe_mape(y_true, y_pred)
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}


def enforce_valid_feature_rows(df, features, sentinel_values=None, feature_ranges=None):
    if not features:
        return df
    cols = [c for c in features if c in df.columns]
    if not cols:
        return df

    sentinel_values = list(sentinel_values or [])
    feature_ranges = dict(feature_ranges or {})

    df2 = df.copy()
    for c in cols:
        df2[c] = pd.to_numeric(df2[c], errors="coerce")

    arr = df2[cols].to_numpy(dtype=float, copy=False)
    keep_mask = np.isfinite(arr).all(axis=1)

    if sentinel_values:
        sent_bad = np.zeros(len(df2), dtype=bool)
        for v in sentinel_values:
            sent_bad |= (arr == float(v)).any(axis=1)
        keep_mask &= ~sent_bad

    if feature_ranges:
        range_mask = np.ones(len(df2), dtype=bool)
        for c in cols:
            rng = feature_ranges.get(c, None)
            if rng is None:
                continue
            lo, hi = rng if isinstance(rng, (tuple, list)) else (None, None)
            col = df2[c].to_numpy(dtype=float, copy=False)
            m = np.isfinite(col)
            if lo is not None:
                m &= (col >= float(lo))
            if hi is not None:
                m &= (col <= float(hi))
            range_mask &= m
        keep_mask &= range_mask

    kept = int(keep_mask.sum())
    dropped = int(len(df2) - kept)
    if dropped > 0:
        print(f"[ValidFilter] kept {kept}/{len(df2)}; dropped {dropped} rows due to invalid required field(s).")
    return df2.loc[keep_mask].copy()


def get_numeric_variance_safe(series_like):
    x = pd.to_numeric(series_like, errors="coerce")
    if x.notna().sum() < 10:
        return np.nan
    return float(np.nanvar(x.to_numpy(dtype=float)))


def prepare_time_features(df):
    out = df.copy()

    out[TIME_COL] = pd.to_numeric(out[TIME_COL], errors="coerce")
    out[MONTH_COL] = pd.to_numeric(out[MONTH_COL], errors="coerce")

    if "month_sin" not in out.columns:
        out["month_sin"] = np.sin(2.0 * np.pi * (out[MONTH_COL] - 1.0) / 12.0)
    if "month_cos" not in out.columns:
        out["month_cos"] = np.cos(2.0 * np.pi * (out[MONTH_COL] - 1.0) / 12.0)

    if "year_norm" not in out.columns:
        y = pd.to_numeric(out[TIME_COL], errors="coerce").to_numpy(dtype=float)
        out["year_norm"] = (y - YEAR_START) / max(1.0, (YEAR_END - YEAR_START))

    if "months_since_1950" not in out.columns:
        yy = pd.to_numeric(out[TIME_COL], errors="coerce").to_numpy(dtype=float)
        mm = pd.to_numeric(out[MONTH_COL], errors="coerce").to_numpy(dtype=float)
        out["months_since_1950"] = (yy - 1950.0) * 12.0 + (mm - 1.0)

    return out


def prepare_geo_features(df):
    out = df.copy()

    lat = pd.to_numeric(out[LAT_COL], errors="coerce").to_numpy(dtype=float, copy=False)
    lon = pd.to_numeric(out[LON_COL], errors="coerce").to_numpy(dtype=float, copy=False)

    lon_m180_180 = ((lon + 180.0) % 360.0) - 180.0
    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon_m180_180)

    if "geo_nx" not in out.columns:
        out["geo_nx"] = np.cos(lat_rad) * np.cos(lon_rad)
    if "geo_ny" not in out.columns:
        out["geo_ny"] = np.cos(lat_rad) * np.sin(lon_rad)
    if "geo_nz" not in out.columns:
        out["geo_nz"] = np.sin(lat_rad)

    return out


def is_thermal_removed_target():
    return FINAL_TARGET == "r_star_T"


def is_direct_pco2_target():
    return FINAL_TARGET == "pCO2"


def target_requires_prior_field():
    return ((FINAL_TARGET == "pCO2") and ENABLE_PRIOR_BASELINE) or INCLUDE_PRIOR_AS_PREDICTOR


def target_requires_prior_clim_field():
    return is_thermal_removed_target()


def target_requires_delta_t_field():
    return is_thermal_removed_target()


def get_target_metric_space_name():
    return FINAL_TARGET


def get_target_plot_labels():
    if FINAL_TARGET == "r_star_T":
        return (
            r"Predicted $r^{*}_{T}$ (ensemble mean)",
            r"Observed $r^{*}_{T}$",
            "r_star_T target-space test panel (ensemble mean)",
            "test_panel_target_r_star_T_ensemble_mean.png",
        )
    return (
        r"Predicted pCO$_2$ (ensemble mean)",
        r"Observed pCO$_2$",
        "Direct pCO2 target-space test panel (ensemble mean)",
        "test_panel_target_pCO2_direct_ensemble_mean.png",
    )


def reconstruct_pco2_from_target(df_part, target_pred):
    target_pred = np.asarray(target_pred, float)

    if FINAL_TARGET == "r_star_T":
        offset = pd.to_numeric(df_part[FINAL_RECON_OFFSET_NAME], errors="coerce").to_numpy(dtype=float, copy=False)
        return offset + target_pred

    if FINAL_TARGET == "pCO2":
        return target_pred.copy()

    raise ValueError(f"Unsupported FINAL_TARGET={FINAL_TARGET}")


def get_reconstruction_mode():
    if FINAL_TARGET == "r_star_T":
        return "pCO2 = (pCO2_prior_clim + delta_pCO2_T) + r_star_T_pred"
    if FINAL_TARGET == "pCO2":
        return "pCO2 = direct prediction"
    return "unknown"


def get_baseline_mode_description():
    if not ENABLE_PRIOR_BASELINE:
        return "disabled"
    if FINAL_TARGET == "r_star_T":
        return "baseline pCO2 = pCO2_prior_clim + delta_pCO2_T"
    if FINAL_TARGET == "pCO2":
        return "baseline pCO2 = pCO2_prior"
    return "unknown"


def get_modeling_mode_description():
    if ENABLE_ZONE_MODELING:
        return f"zonewise (one model per {ZONE_COL})"
    return "global (single model, ignore Zone during fitting)"


def resolve_or_build_core_fields(df):
    out = df.copy()

    label_col = find_first_existing(out.columns, LABEL_CANDIDATES, required=True, field_desc="pCO2 label field")
    prior_col = find_first_existing(
        out.columns,
        PRIOR_CANDIDATES,
        required=target_requires_prior_field(),
        field_desc="pCO2_prior field"
    )
    prior_clim_col = find_first_existing(
        out.columns,
        PRIOR_CLIM_CANDIDATES,
        required=target_requires_prior_clim_field(),
        field_desc="pCO2 prior climatology field"
    )
    r_star_t_col = find_first_existing(out.columns, R_STAR_T_CANDIDATES, required=False, field_desc="r_star_T field")
    delta_t_col = find_first_existing(
        out.columns,
        DELTA_PCO2_T_CANDIDATES,
        required=target_requires_delta_t_field(),
        field_desc="delta_pCO2_T field"
    )
    date_col = find_first_existing(out.columns, DATE_CANDIDATES, required=False, field_desc="date field")

    out[FINAL_LABEL_NAME] = pd.to_numeric(out[label_col], errors="coerce")

    if prior_col is not None:
        out[FINAL_PRIOR_NAME] = pd.to_numeric(out[prior_col], errors="coerce")
    else:
        out[FINAL_PRIOR_NAME] = np.nan

    if prior_clim_col is not None:
        out[FINAL_PRIOR_CLIM_NAME] = pd.to_numeric(out[prior_clim_col], errors="coerce")
    else:
        out[FINAL_PRIOR_CLIM_NAME] = np.nan

    if delta_t_col is not None:
        out[FINAL_DELTA_T_NAME] = pd.to_numeric(out[delta_t_col], errors="coerce")
    else:
        out[FINAL_DELTA_T_NAME] = np.nan

    if FINAL_TARGET == "r_star_T":
        thermal_prior = out[FINAL_PRIOR_CLIM_NAME] + out[FINAL_DELTA_T_NAME]
        if r_star_t_col is not None:
            out[FINAL_TARGET] = pd.to_numeric(out[r_star_t_col], errors="coerce")
            target_source_col = r_star_t_col
        else:
            if (prior_clim_col is None) or (delta_t_col is None):
                raise KeyError("FINAL_TARGET='r_star_T' requires (pCO2_prior_clim and delta_pCO2_T) or a precomputed r_star_T column.")
            out[FINAL_TARGET] = out[FINAL_LABEL_NAME] - thermal_prior
            target_source_col = "computed_from_label_minus_(pCO2_prior_clim_plus_delta_pCO2_T)"
        out[FINAL_RECON_OFFSET_NAME] = thermal_prior
        out[FINAL_BASELINE_PCO2_NAME] = thermal_prior

    elif FINAL_TARGET == "pCO2":
        out[FINAL_TARGET] = out[FINAL_LABEL_NAME]
        target_source_col = label_col
        out[FINAL_RECON_OFFSET_NAME] = 0.0
        if prior_col is not None:
            out[FINAL_BASELINE_PCO2_NAME] = out[FINAL_PRIOR_NAME]
        else:
            out[FINAL_BASELINE_PCO2_NAME] = np.nan

    else:
        raise ValueError(f"Unsupported FINAL_TARGET={FINAL_TARGET}")

    if date_col is not None:
        out["_Date_used"] = pd.to_datetime(out[date_col], errors="coerce")
    else:
        yy = pd.to_numeric(out[TIME_COL], errors="coerce")
        mm = pd.to_numeric(out[MONTH_COL], errors="coerce")
        out["_Date_used"] = pd.to_datetime(dict(year=yy, month=mm, day=15), errors="coerce")

    resolved = {
        "label_col": label_col,
        "prior_col": prior_col,
        "prior_clim_col": prior_clim_col,
        "r_star_T_col": r_star_t_col,
        "delta_pCO2_T_col": delta_t_col,
        "date_col": date_col,
        "target_used": FINAL_TARGET,
        "target_source_col": target_source_col,
        "reconstruction_mode": get_reconstruction_mode(),
        "baseline_mode": get_baseline_mode_description(),
        "modeling_mode": MODELING_MODE,
        "modeling_mode_desc": get_modeling_mode_description(),
    }
    return out, resolved


def plot_test_panel(y_true, y_pred, out_path,
                    xlab="Predicted", ylab="Observed", title=None):
    if not ENABLE_PLOTTING:
        return

    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if y_true.size == 0:
        return

    resid = y_true - y_pred

    try:
        import scipy.stats as st
        skew = float(st.skew(resid, nan_policy="omit"))
        kurt = float(st.kurtosis(resid, nan_policy="omit", fisher=False))
    except Exception:
        skew, kurt = np.nan, np.nan

    met = compute_metrics_dict(y_true, y_pred)

    fig = plt.figure(figsize=(12.8, 5.4))
    ax1 = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2)

    hb = ax1.hexbin(y_pred, y_true, gridsize=150, mincnt=1, linewidths=0)
    arr = hb.get_array()
    if arr.size:
        vmax = float(np.nanpercentile(arr, 99.0))
        hb.set_clim(0, vmax)

    dmax = float(max(np.nanmax(y_pred), np.nanmax(y_true)))
    dmin = float(min(np.nanmin(y_pred), np.nanmin(y_true)))
    ax1.plot([dmin, dmax], [dmin, dmax], ls="--", lw=1.2, color="0.55")
    ax1.set_xlabel(xlab)
    ax1.set_ylabel(ylab)
    ax1.grid(alpha=0.22, lw=0.5)
    if title is not None:
        ax1.set_title(title)
    ax1.text(
        0.03, 0.95,
        f"R2={met['R2']:.3f}\nMAE={met['MAE']:.3f}\nRMSE={met['RMSE']:.3f}\nMAPE={met['MAPE']:.3f}%",
        transform=ax1.transAxes, va="top", ha="left"
    )

    cb = fig.colorbar(hb, ax=ax1, pad=0.01)
    cb.set_label("Counts")

    ax2.hist(resid, bins=80, alpha=0.9, edgecolor="none")
    ax2.set_xlabel("Residual (Obs - Pred)")
    ax2.set_ylabel("Counts")
    ax2.grid(alpha=0.20, lw=0.5)
    ax2.text(
        0.03, 0.95,
        f"skew={skew:+.3f}\nkurt={kurt:.3f}",
        transform=ax2.transAxes, va="top", ha="left"
    )

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def summarize_uncertainty(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {"mean": np.nan, "median": np.nan, "p95": np.nan, "max": np.nan}
    return {
        "mean": float(np.nanmean(x)),
        "median": float(np.nanmedian(x)),
        "p95": float(np.nanpercentile(x, 95.0)),
        "max": float(np.nanmax(x)),
    }


def evaluate_prior_only_baseline(train_df, test_df):
    rows = []
    calib_train = {}
    calib_test = {}

    def _eval_one(df_part, split_name):
        if df_part is None or len(df_part) == 0:
            return None, {}

        y_true = pd.to_numeric(df_part[FINAL_LABEL_NAME], errors="coerce").to_numpy(dtype=float, copy=False)
        y_pred = pd.to_numeric(df_part[FINAL_BASELINE_PCO2_NAME], errors="coerce").to_numpy(dtype=float, copy=False)

        mask = np.isfinite(y_true) & np.isfinite(y_pred)
        if not np.any(mask):
            return None, {}

        met = compute_metrics_dict(y_true[mask], y_pred[mask])
        cal = calibration_stats(y_true[mask], y_pred[mask])

        row = [
            "PRIOR_BASELINE",
            split_name,
            "prior_only_pCO2",
            met["MSE"],
            met["RMSE"],
            met["MAE"],
            met["R2"],
            met["MAPE"],
        ]
        return row, cal

    row_tr, calib_train = _eval_one(train_df, "train")
    if row_tr is not None:
        rows.append(row_tr)

    row_te, calib_test = _eval_one(test_df, "test")
    if row_te is not None:
        rows.append(row_te)

    return rows, calib_train, calib_test


def normalize_zone_value(v):
    """
    将 Zone 值规范化为稳定的字符串 key，便于分组和建目录。
    支持整数、浮点、字符串。
    """
    if pd.isna(v):
        return np.nan

    try:
        fv = float(v)
        if not np.isfinite(fv):
            return np.nan
        if abs(fv - round(fv)) < 1e-9:
            return str(int(round(fv)))
        return f"{fv:g}"
    except Exception:
        s = str(v).strip()
        return s if s != "" else np.nan


def build_zone_key_series(series):
    return series.apply(normalize_zone_value)


def zone_dir_name(zone_key):
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", str(zone_key))
    return f"Zone_{safe}"


def sort_zone_keys(zone_keys):
    def _k(z):
        try:
            return (0, float(z))
        except Exception:
            return (1, str(z))
    return sorted(zone_keys, key=_k)


# =========================================================
# 5. Feature engineering
# =========================================================
def compute_feature_stats(train_df, chla_offset):
    stats = {}

    def _mean_of(colname, fallback_series=None):
        if fallback_series is not None:
            x = pd.to_numeric(fallback_series, errors="coerce")
        elif colname in train_df.columns:
            x = pd.to_numeric(train_df[colname], errors="coerce")
        else:
            return np.nan
        return float(np.nanmean(x.to_numpy(dtype=float))) if x.notna().sum() > 0 else np.nan

    log10_chla_train = get_log10_chla(train_df, chla_offset=chla_offset)

    stats["SST_mean"] = _mean_of("SST")
    stats["SSS_mean"] = _mean_of("SSS")
    stats["DO_mean"] = _mean_of("DO")
    stats["log10_Chla_mean"] = _mean_of("log10_Chla", fallback_series=log10_chla_train)
    stats["NINO34_mean"] = _mean_of("NINO34")
    return stats


def transform_features(df_part, feature_cols, chla_offset, feature_stats):
    X = pd.DataFrame(index=df_part.index)

    sst = pd.to_numeric(df_part["SST"], errors="coerce") if "SST" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    sss = pd.to_numeric(df_part["SSS"], errors="coerce") if "SSS" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    do = pd.to_numeric(df_part["DO"], errors="coerce") if "DO" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    sst_anom = pd.to_numeric(df_part["SST_anom"], errors="coerce") if "SST_anom" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    do_anom = pd.to_numeric(df_part["DO_anom"], errors="coerce") if "DO_anom" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    nino34 = pd.to_numeric(df_part["NINO34"], errors="coerce") if "NINO34" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    log10_chla = get_log10_chla(df_part, chla_offset)

    sst_c = sst - float(feature_stats.get("SST_mean", np.nan))
    sss_c = sss - float(feature_stats.get("SSS_mean", np.nan))
    do_c = do - float(feature_stats.get("DO_mean", np.nan))
    chla_c = log10_chla - float(feature_stats.get("log10_Chla_mean", np.nan))

    for c in feature_cols:
        if c == "log10_Chla":
            X[c] = log10_chla
        elif c == "SSTc_x_DOc":
            X[c] = sst_c * do_c
        elif c == "SSTc_x_SSSc":
            X[c] = sst_c * sss_c
        elif c == "SSTanom_x_DOanom":
            X[c] = sst_anom * do_anom
        elif c == "SSTanom_x_NINO34":
            X[c] = sst_anom * nino34
        elif c == "DOc_x_log10Chla":
            X[c] = do_c * chla_c
        else:
            if c in df_part.columns:
                X[c] = pd.to_numeric(df_part[c], errors="coerce")
            else:
                X[c] = np.nan
    return X


def select_available_features_from_train(train_df, chla_offset, feature_stats):
    X_all = transform_features(train_df, ALL_CANDIDATE_FEATURES, chla_offset, feature_stats)

    usable = []
    for c in ALL_CANDIDATE_FEATURES:
        if c not in X_all.columns:
            continue
        x = pd.to_numeric(X_all[c], errors="coerce")
        n_valid = int(x.notna().sum())
        var = get_numeric_variance_safe(x)
        if n_valid >= 50 and np.isfinite(var) and var > 0:
            usable.append(c)

    keep = [c for c in ALWAYS_KEEP if c in usable]
    others = [c for c in usable if c not in keep]
    return keep + others, X_all


# =========================================================
# 6. Inverse-density weights
# =========================================================
def build_density_sample_weight(df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    w = np.ones(n, dtype=float)
    if not ENABLE_DENSITY_WEIGHTING:
        return w

    if not {LAT_COL, LON_COL, TIME_COL}.issubset(df.columns):
        print("[DensityWeight] Missing Latitude/Longitude/Year. Fallback to uniform weights.")
        return w

    res_deg = float(WEIGHT_GRID_RESOLUTION_DEG)
    res_yr = float(WEIGHT_TIME_RESOLUTION_YR)

    lat = pd.to_numeric(df[LAT_COL], errors="coerce").to_numpy(dtype=float, copy=False)
    lon = pd.to_numeric(df[LON_COL], errors="coerce").to_numpy(dtype=float, copy=False)
    yr = pd.to_numeric(df[TIME_COL], errors="coerce").to_numpy(dtype=float, copy=False)

    key_ok = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(yr)
    if not np.any(key_ok):
        print("[DensityWeight] All Latitude/Longitude/Year invalid. Fallback to uniform weights.")
        return w

    idx_ok = np.where(key_ok)[0]
    lon_ok = ((lon[idx_ok] + 180.0) % 360.0) - 180.0

    lat_bin = np.floor(lat[idx_ok] / res_deg).astype(np.int64)
    lon_bin = np.floor(lon_ok / res_deg).astype(np.int64)
    time_bin = np.floor(yr[idx_ok] / res_yr).astype(np.int64)

    t0 = int(time_bin.min())
    la0 = int(lat_bin.min())
    lo0 = int(lon_bin.min())

    tb = (time_bin - t0).astype(np.int64)
    lab = (lat_bin - la0).astype(np.int64)
    lob = (lon_bin - lo0).astype(np.int64)

    n_lat = int(lab.max()) + 1
    n_lon = int(lob.max()) + 1

    key = (tb * np.int64(n_lat) + lab) * np.int64(n_lon) + lob
    _, inv = np.unique(key, return_inverse=True)
    counts = np.bincount(inv).astype(float)
    cnt_each = counts[inv]

    ww = 1.0 / np.sqrt(np.maximum(cnt_each, 1.0))

    if ENABLE_TRANSITION_PERIOD_PENALTY:
        yy = yr[idx_ok]
        m = (yy >= float(PENALTY_YEAR_START)) & (yy <= float(PENALTY_YEAR_END))
        if np.any(m):
            ww[m] *= float(PENALTY_FACTOR)

    w[idx_ok] = ww

    mean_w = float(np.nanmean(w)) if np.isfinite(w).any() else 1.0
    if (not np.isfinite(mean_w)) or mean_w <= 0:
        mean_w = 1.0
    w = w / mean_w
    return w


# =========================================================
# 7. Decade-block KFold
# =========================================================
class DecadeBlockKFold:
    def __init__(self, n_splits=5, random_state=42):
        if n_splits < 2:
            raise ValueError("n_splits must be >= 2")
        self.n_splits = int(n_splits)
        self.random_state = int(random_state)

    def split(self, X, y=None, groups=None):
        n = len(X)
        if groups is None or len(groups) != n:
            rng = np.random.RandomState(self.random_state)
            idx = np.arange(n)
            rng.shuffle(idx)
            folds = np.array_split(idx, min(self.n_splits, max(2, n)))
            for f in folds:
                va = np.sort(f)
                tr = np.sort(np.setdiff1d(idx, va, assume_unique=False))
                yield tr, va
            return

        years = np.asarray(groups, dtype=float)
        decade = np.where(np.isfinite(years), (np.floor(years / 10.0) * 10).astype(int), -999999)

        uniq_dec = np.unique(decade)
        if len(uniq_dec) < 2:
            rng = np.random.RandomState(self.random_state)
            idx = np.arange(n)
            rng.shuffle(idx)
            folds = np.array_split(idx, min(self.n_splits, max(2, n)))
            for f in folds:
                va = np.sort(f)
                tr = np.sort(np.setdiff1d(idx, va, assume_unique=False))
                yield tr, va
            return

        rng = np.random.RandomState(self.random_state)
        rng.shuffle(uniq_dec)
        k_eff = min(self.n_splits, len(uniq_dec))
        dec_folds = np.array_split(uniq_dec, k_eff)

        for dec_va in dec_folds:
            dec_va_set = set(dec_va.tolist())
            va_mask = np.isin(decade, list(dec_va_set))
            va_idx = np.where(va_mask)[0]
            tr_idx = np.where(~va_mask)[0]
            yield tr_idx, va_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        if groups is None:
            return self.n_splits
        years = np.asarray(groups, dtype=float)
        decade = np.where(np.isfinite(years), (np.floor(years / 10.0) * 10).astype(int), -999999)
        uniq_dec = np.unique(decade)
        return min(self.n_splits, max(2, len(uniq_dec)))


# =========================================================
# 8. Model factories / FS
# =========================================================
def make_cb_regressor_for_fs():
    return CatBoostRegressor(
        iterations=700,
        depth=8,
        learning_rate=0.06,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_SEED,
        thread_count=N_THREADS,
        verbose=False
    )


def _weighted_mse(y_true, y_pred, w):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    w = np.asarray(w, float)
    m = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(w)
    if not np.any(m):
        return np.nan
    y_true = y_true[m]
    y_pred = y_pred[m]
    w = w[m]
    sw = np.sum(w)
    if sw <= 0 or not np.isfinite(sw):
        return np.nan
    err = y_pred - y_true
    return float(np.sum(w * err * err) / sw)


def permutation_importance_cv_weighted(X_df, y, years, sample_weight,
                                       n_splits=5, n_repeats=3, max_val_samples=12000):
    cols = list(X_df.columns)
    p = len(cols)
    scores = np.zeros(p, dtype=float)

    cv = DecadeBlockKFold(n_splits=n_splits, random_state=RANDOM_SEED)
    y = np.asarray(y)
    sw_all = np.asarray(sample_weight, float)
    rng_global = np.random.RandomState(RANDOM_SEED)

    for tr_idx, va_idx in cv.split(X_df, y, groups=years):
        X_tr = X_df.iloc[tr_idx].values
        y_tr = y[tr_idx]
        sw_tr = sw_all[tr_idx]

        if len(va_idx) > int(max_val_samples):
            va_idx = rng_global.choice(va_idx, size=int(max_val_samples), replace=False)

        X_va0 = X_df.iloc[va_idx].values
        y_va = y[va_idx]
        sw_va = sw_all[va_idx]

        est = make_cb_regressor_for_fs()
        est.fit(X_tr, y_tr, sample_weight=sw_tr)

        base_pred = est.predict(X_va0)
        base_mse = _weighted_mse(y_va, base_pred, sw_va)
        if not np.isfinite(base_mse):
            continue

        for j in range(p):
            inc = 0.0
            for _ in range(int(n_repeats)):
                X_perm = X_va0.copy()
                perm_idx = rng_global.permutation(X_perm.shape[0])
                X_perm[:, j] = X_perm[perm_idx, j]
                pred = est.predict(X_perm)
                mse_perm = _weighted_mse(y_va, pred, sw_va)
                if np.isfinite(mse_perm):
                    inc += max(0.0, mse_perm - base_mse)
            scores[j] += inc / max(1, int(n_repeats))

    denom = cv.get_n_splits(X_df, y, groups=years)
    if denom <= 0:
        denom = 1
    scores /= float(denom)
    return pd.Series(scores, index=cols).sort_values(ascending=False)


def rfecv_with_decade_blocks(X_df, y, years, sample_weight,
                             always_keep=None,
                             min_feats=8, step=1, scoring="neg_root_mean_squared_error"):
    always_keep = list(always_keep) if always_keep else []

    cols_all = list(X_df.columns)
    p_total = len(cols_all)

    keep_in = [c for c in cols_all if c not in always_keep]
    X_core = X_df[keep_in].copy()
    p_core = len(keep_in)

    if USE_ADAPTIVE_RFECV_MIN:
        min_total = max(RFECV_MIN_ABS, int(np.ceil(p_total * RFECV_MIN_RATIO)))
        min_total = min(min_total, max(1, p_total - 1))
        min_total = min(min_total, max(1, min_feats, RFECV_MIN_FEATS))
    else:
        min_total = min(max(1, min_feats), max(1, p_total - 1))

    min_core = max(1, min_total - len(always_keep))
    min_core = min(min_core, max(1, p_core - 1))

    info = {
        "p_total": int(p_total),
        "p_core": int(p_core),
        "min_total_used": int(min_total),
        "min_core_used": int(min_core),
        "policy": ("adaptive" if USE_ADAPTIVE_RFECV_MIN else "fixed"),
        "ratio": float(RFECV_MIN_RATIO),
        "abs_min": int(RFECV_MIN_ABS),
        "fixed_min": int(RFECV_MIN_FEATS),
        "rfecv_weighted_supported": None
    }

    if p_core <= 1 or min_core >= p_core:
        final = list(dict.fromkeys(always_keep + keep_in))
        return final, None, info

    est = make_cb_regressor_for_fs()
    cv = DecadeBlockKFold(n_splits=PERM_N_SPLITS, random_state=RANDOM_SEED)

    rfecv = RFECV(
        estimator=est,
        step=step,
        cv=cv,
        scoring=scoring,
        min_features_to_select=min_core,
        n_jobs=1
    )

    try:
        rfecv.fit(
            X_core.values, y,
            groups=np.asarray(years),
            sample_weight=np.asarray(sample_weight, float)
        )
        info["rfecv_weighted_supported"] = True
    except TypeError:
        print("[RFECV] 当前 sklearn 版本不支持 RFECV.fit(sample_weight=...)，降级为无权重 RFECV。")
        rfecv.fit(X_core.values, y, groups=np.asarray(years))
        info["rfecv_weighted_supported"] = False

    mask = rfecv.support_
    selected_core = list(np.array(keep_in)[mask])
    final = list(dict.fromkeys(always_keep + selected_core))
    return final, rfecv, info


# =========================================================
# 9. Optuna objective
# =========================================================
def objective(trial, X, y, years, sample_weights):
    X = np.asarray(X)
    y = np.asarray(y)
    years = np.asarray(years)
    sw_all = np.asarray(sample_weights, float)

    params = {
        "iterations": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_int("l2_leaf_reg", 5, 20),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.1, 0.8),
        "random_strength": trial.suggest_float("random_strength", 0.1, 2.0),
        "bootstrap_type": "Bayesian",
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": RANDOM_SEED,
        "od_type": "Iter",
        "od_wait": 80,
        "use_best_model": True,
        "verbose": False
    }

    cv = DecadeBlockKFold(n_splits=5, random_state=RANDOM_SEED)
    rmses, best_iters = [], []

    for tr_idx, va_idx in cv.split(X, y, groups=years):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        sw_tr = sw_all[tr_idx]
        sw_va = sw_all[va_idx]

        eval_pool = Pool(X_va, y_va, weight=sw_va)
        model = CatBoostRegressor(**params, thread_count=N_THREADS)
        model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=eval_pool)

        preds = model.predict(eval_pool)
        err = preds - y_va

        denom = np.sum(sw_va)
        if denom <= 0 or not np.isfinite(denom):
            wrmse = float(np.sqrt(np.mean(err * err)))
        else:
            wrmse = float(np.sqrt(np.sum(sw_va * err * err) / denom))

        rmses.append(wrmse)
        try:
            best_iters.append(int(model.get_best_iteration()))
        except Exception:
            best_iters.append(int(params["iterations"]))

    trial.set_user_attr("best_iters", best_iters)
    trial.set_user_attr(
        "median_best_iter",
        int(np.median(best_iters)) if best_iters else int(params["iterations"])
    )
    return float(np.mean(rmses)) if rmses else np.nan


# =========================================================
# 10. Ensemble helpers
# =========================================================
def bootstrap_member_indices(n_rows, seed, frac=1.0):
    rng = np.random.default_rng(int(seed))
    n_boot = int(max(1, round(float(frac) * int(n_rows))))
    return rng.integers(0, int(n_rows), size=n_boot, endpoint=False)


def train_one_member(X_tr, y_tr, sw_tr, final_params, member_seed,
                     member_idx=None, save_path=None):
    params = dict(final_params)
    params["random_seed"] = int(member_seed)

    model = CatBoostRegressor(**params, thread_count=N_THREADS)

    if member_idx is not None:
        X_fit = X_tr[member_idx]
        y_fit = y_tr[member_idx]
        sw_fit = sw_tr[member_idx]
    else:
        X_fit = X_tr
        y_fit = y_tr
        sw_fit = sw_tr

    model.fit(X_fit, y_fit, sample_weight=sw_fit)

    if save_path is not None:
        model.save_model(save_path)

    return model


def collect_member_predictions(models, X):
    preds = []
    for mdl in models:
        preds.append(np.asarray(mdl.predict(X), float))
    if len(preds) == 0:
        return np.empty((len(X), 0), dtype=float)
    return np.column_stack(preds)


# =========================================================
# 11. Scope training core (shared by zonewise/global)
# =========================================================
def run_scope_ensemble_model(train_df, test_df, selected_candidates,
                             resolved_cols,
                             chla_offset, feature_stats, X_tr_all_df,
                             output_dir,
                             scope_name="model_scope",
                             zone_key=None):
    """
    Shared training core for one modeling scope.
    A scope can be:
      - one Zone (zonewise mode)
      - the whole globe (global mode)

    The methodological chain is identical across scopes.
    """

    os.makedirs(output_dir, exist_ok=True)
    member_dir = os.path.join(output_dir, "ensemble_members")
    os.makedirs(member_dir, exist_ok=True)

    if len(train_df) < MIN_SAMPLES:
        print(f"[{scope_name}] skipped: train samples < MIN_SAMPLES ({len(train_df)} < {MIN_SAMPLES})")
        empty_pred = pd.DataFrame(index=test_df.index)
        return [], None, empty_pred

    sw = train_df["sample_weight"].values if "sample_weight" in train_df.columns else np.ones(len(train_df), dtype=float)

    candidates = [c for c in selected_candidates if c in X_tr_all_df.columns]
    if len(candidates) == 0:
        raise ValueError(f"[{scope_name}] No usable predictor is available.")

    always_keep_present = [f for f in ALWAYS_KEEP if f in candidates]

    y_tr = train_df[FINAL_TARGET].values
    years = train_df[TIME_COL].values

    pi_score = None
    rfecv_info = None

    if ENABLE_FEATURE_SELECTION:
        X_for_pi = X_tr_all_df[candidates].copy()

        pi_score = permutation_importance_cv_weighted(
            X_for_pi, y_tr, years, sw,
            n_splits=PERM_N_SPLITS,
            n_repeats=PERM_N_REPEATS,
            max_val_samples=PERM_MAX_VAL_SAMPLES
        )

        if PERM_TOPK is None:
            k_auto = max(10, int(2 * sqrt(len(pi_score))))
            top_feats = list(pi_score.index[:k_auto])
        else:
            top_feats = list(pi_score.index[:min(int(PERM_TOPK), len(pi_score))])

        rfecv_candidates = list(dict.fromkeys(always_keep_present + top_feats))
        X_rfecv = X_for_pi[rfecv_candidates].copy()

        selected_feats, _, rfecv_info = rfecv_with_decade_blocks(
            X_rfecv, y_tr, years, sw,
            always_keep=always_keep_present,
            min_feats=max(RFECV_MIN_FEATS, len(always_keep_present)),
            step=RFECV_STEP,
            scoring=RFECV_SCORING
        )

        selected_feats_sorted = [f for f in pi_score.index if f in selected_feats]
        for f in always_keep_present:
            if f not in selected_feats_sorted:
                selected_feats_sorted = [f] + selected_feats_sorted
    else:
        selected_feats_sorted = [f for f in candidates]
        for f in always_keep_present:
            if f not in selected_feats_sorted:
                selected_feats_sorted.append(f)

    X_tr = X_tr_all_df[selected_feats_sorted].values

    if len(test_df) > 0:
        X_te_df = transform_features(test_df, selected_feats_sorted, chla_offset, feature_stats)
        X_te_df = X_te_df.reindex(columns=selected_feats_sorted, fill_value=np.nan)
        X_te = X_te_df.values
    else:
        X_te = None

    # -------------------------
    # Optuna
    # -------------------------
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED))
    study.optimize(
        lambda trial: objective(trial, X_tr, y_tr, years, sw),
        n_trials=N_TRIALS_OPTUNA,
        show_progress_bar=False
    )

    best_params = study.best_trial.params.copy()
    best_iters = study.best_trial.user_attrs.get("best_iters", [])
    median_best_iter = int(np.median(best_iters)) if len(best_iters) > 0 else int(best_params.get("iterations", 800))

    best_params.update({
        "bootstrap_type": "Bayesian",
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": RANDOM_SEED,
    })

    final_iters = int(np.clip(median_best_iter, 200, 2000))
    final_params = {**best_params, "iterations": final_iters, "verbose": False}
    for k in ["od_type", "od_wait", "use_best_model"]:
        final_params.pop(k, None)

    # -------------------------
    # Ensemble training
    # -------------------------
    n_members = int(ENSEMBLE_SIZE) if ENABLE_ENSEMBLE else 1
    member_models = []
    member_records = []
    fi_rows = []

    for i in range(n_members):
        member_seed = int(ENSEMBLE_BASE_SEED + i * 101)
        member_idx = None
        if ENABLE_ENSEMBLE and ENSEMBLE_BOOTSTRAP_ROWS:
            member_idx = bootstrap_member_indices(len(X_tr), member_seed, frac=ENSEMBLE_BOOTSTRAP_FRAC)

        save_path = os.path.join(member_dir, f"member_{i:02d}.cbm") if SAVE_MEMBER_MODELS else None
        model = train_one_member(
            X_tr=X_tr,
            y_tr=y_tr,
            sw_tr=sw,
            final_params=final_params,
            member_seed=member_seed,
            member_idx=member_idx,
            save_path=save_path
        )
        member_models.append(model)

        fi = np.asarray(model.get_feature_importance(), float)
        fi_rows.append(fi)
        member_records.append({
            "member": i,
            "seed": member_seed,
            "n_rows_fit": int(len(member_idx)) if member_idx is not None else int(len(X_tr)),
            "n_unique_rows_fit": int(len(np.unique(member_idx))) if member_idx is not None else int(len(X_tr)),
            "model_path": save_path if save_path is not None else ""
        })

    member_info_df = pd.DataFrame(member_records)
    member_info_df.to_csv(os.path.join(output_dir, "ensemble_member_info.csv"), index=False)

    fi_arr = np.vstack(fi_rows) if len(fi_rows) > 0 else np.empty((0, len(selected_feats_sorted)), dtype=float)
    fi_summary = pd.DataFrame({
        "Feature": selected_feats_sorted,
        "Importance_mean": np.nanmean(fi_arr, axis=0) if fi_arr.size else np.nan,
        "Importance_std": np.nanstd(fi_arr, axis=0) if fi_arr.size else np.nan,
    }).sort_values("Importance_mean", ascending=False)
    fi_summary.to_csv(os.path.join(output_dir, "feature_importance_ensemble.csv"), index=False)

    # -------------------------
    # Train/test prediction
    # -------------------------
    tr_member_pred = collect_member_predictions(member_models, X_tr)
    tr_pred_mean = np.nanmean(tr_member_pred, axis=1) if tr_member_pred.size else np.full(len(X_tr), np.nan)
    tr_pred_std = np.nanstd(tr_member_pred, axis=1, ddof=0) if tr_member_pred.size else np.full(len(X_tr), np.nan)

    if X_te is not None and len(test_df) > 0:
        te_member_pred = collect_member_predictions(member_models, X_te)
        te_pred_mean = np.nanmean(te_member_pred, axis=1) if te_member_pred.size else np.full(len(X_te), np.nan)
        te_pred_std = np.nanstd(te_member_pred, axis=1, ddof=0) if te_member_pred.size else np.full(len(X_te), np.nan)
    else:
        te_member_pred = np.empty((0, 0), dtype=float)
        te_pred_mean = None
        te_pred_std = None

    # -------------------------
    # Metrics
    # -------------------------
    metrics_rows = []

    if ENABLE_PRIOR_BASELINE:
        prior_rows, _, _ = evaluate_prior_only_baseline(train_df, test_df)
        metrics_rows.extend(prior_rows)

    target_space_name = get_target_metric_space_name()

    met_tr_target = compute_metrics_dict(y_tr, tr_pred_mean)
    metrics_rows.append(["ENSEMBLE_MEAN", "train", target_space_name,
                         met_tr_target["MSE"], met_tr_target["RMSE"], met_tr_target["MAE"],
                         met_tr_target["R2"], met_tr_target["MAPE"]])

    pco2_true_tr = train_df[FINAL_LABEL_NAME].values
    pco2_pred_tr_mean = reconstruct_pco2_from_target(train_df, tr_pred_mean)
    met_tr_pco2 = compute_metrics_dict(pco2_true_tr, pco2_pred_tr_mean)
    if FINAL_TARGET != "pCO2":
        metrics_rows.append(["ENSEMBLE_MEAN", "train", "reconstructed_pCO2",
                             met_tr_pco2["MSE"], met_tr_pco2["RMSE"], met_tr_pco2["MAE"],
                             met_tr_pco2["R2"], met_tr_pco2["MAPE"]])

    if te_pred_mean is not None and len(test_df) > 0:
        y_te = test_df[FINAL_TARGET].values
        met_te_target = compute_metrics_dict(y_te, te_pred_mean)
        metrics_rows.append(["ENSEMBLE_MEAN", "test", target_space_name,
                             met_te_target["MSE"], met_te_target["RMSE"], met_te_target["MAE"],
                             met_te_target["R2"], met_te_target["MAPE"]])

        pco2_true_te = test_df[FINAL_LABEL_NAME].values
        pco2_pred_te_mean = reconstruct_pco2_from_target(test_df, te_pred_mean)
        met_te_pco2 = compute_metrics_dict(pco2_true_te, pco2_pred_te_mean)
        if FINAL_TARGET != "pCO2":
            metrics_rows.append(["ENSEMBLE_MEAN", "test", "reconstructed_pCO2",
                                 met_te_pco2["MSE"], met_te_pco2["RMSE"], met_te_pco2["MAE"],
                                 met_te_pco2["R2"], met_te_pco2["MAPE"]])

    pd.DataFrame(
        metrics_rows,
        columns=["Group", "Split", "MetricSpace", "MSE", "RMSE", "MAE", "R2", "MAPE"]
    ).to_csv(os.path.join(output_dir, "all_metrics.csv"), index=False)

    # -------------------------
    # Metadata
    # -------------------------
    valid_trials = [t for t in study.trials if (t.state == optuna.trial.TrialState.COMPLETE and t.value is not None)]
    cv_rmses = [t.value for t in valid_trials]
    cv_mean, cv_std = (float(np.mean(cv_rmses)), float(np.std(cv_rmses))) if len(cv_rmses) > 0 else (np.nan, np.nan)

    metadata = {
        "scope_name": scope_name,
        "zone_key": zone_key,
        "run_mode": RUN_MODE,
        "final_target": FINAL_TARGET,
        "target_metric_space": target_space_name,
        "pCO2_reconstruction_mode": get_reconstruction_mode(),
        "prior_baseline_enabled": bool(ENABLE_PRIOR_BASELINE),
        "baseline_mode": get_baseline_mode_description(),
        "modeling_mode": MODELING_MODE,
        "modeling_mode_desc": get_modeling_mode_description(),
        "ensemble_enabled": bool(ENABLE_ENSEMBLE),
        "ensemble_size": int(n_members),
        "ensemble_bootstrap_rows": bool(ENSEMBLE_BOOTSTRAP_ROWS),
        "ensemble_bootstrap_frac": float(ENSEMBLE_BOOTSTRAP_FRAC),
        "features": selected_feats_sorted,
        "feature_engineering_enabled": bool(ENABLE_FEATURE_ENGINEERING),
        "feature_selection_enabled": bool(ENABLE_FEATURE_SELECTION),
        "chla_offset": float(chla_offset) if chla_offset is not None else None,
        "feature_stats": feature_stats,
        "best_params_representative_member": final_params,
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "cv_best_iterations_per_fold": best_iters,
        "final_iterations": int(final_iters),
        "cv_rmse_mean": cv_mean,
        "cv_rmse_std": cv_std,
        "resolved_columns": resolved_cols,
    }

    joblib.dump(metadata, os.path.join(output_dir, "metadata.pkl"))
    with open(os.path.join(output_dir, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    with open(os.path.join(output_dir, "training_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Scope: {scope_name}\n")
        f.write(f"Zone key: {zone_key}\n")
        f.write(f"Modeling mode: {MODELING_MODE}\n")
        f.write(f"Modeling mode desc: {get_modeling_mode_description()}\n")
        f.write(f"Train samples: {len(train_df)}\n")
        f.write(f"Test samples: {len(test_df)}\n")
        f.write(f"Final target: {FINAL_TARGET}\n")
        f.write(f"Features: {selected_feats_sorted}\n")
        f.write(f"Best params: {final_params}\n")
        f.write(f"Best iters per fold: {best_iters}\n")
        f.write(f"CV RMSE mean/std: {cv_mean:.6f} / {cv_std:.6f}\n")

    # -------------------------
    # Plots
    # -------------------------
    if ENABLE_PLOTTING and te_pred_mean is not None and len(test_df) > 0:
        if ENABLE_PRIOR_BASELINE:
            y_true_base_te = pd.to_numeric(test_df[FINAL_LABEL_NAME], errors="coerce").to_numpy(dtype=float, copy=False)
            y_pred_base_te = pd.to_numeric(test_df[FINAL_BASELINE_PCO2_NAME], errors="coerce").to_numpy(dtype=float, copy=False)
            if np.isfinite(y_true_base_te).any() and np.isfinite(y_pred_base_te).any():
                plot_test_panel(
                    y_true_base_te, y_pred_base_te,
                    out_path=os.path.join(output_dir, "test_panel_prior_baseline_pco2.png"),
                    xlab=r"Prior-only predicted pCO$_2$",
                    ylab=r"Observed pCO$_2$",
                    title=f"Prior-only baseline test panel | {scope_name}"
                )

        xlab_t, ylab_t, title_t, fname_t = get_target_plot_labels()
        plot_test_panel(
            test_df[FINAL_TARGET].values, te_pred_mean,
            out_path=os.path.join(output_dir, fname_t),
            xlab=xlab_t,
            ylab=ylab_t,
            title=f"{title_t} | {scope_name}"
        )
        if FINAL_TARGET != "pCO2":
            plot_test_panel(
                test_df[FINAL_LABEL_NAME].values,
                reconstruct_pco2_from_target(test_df, te_pred_mean),
                out_path=os.path.join(output_dir, "test_panel_reconstructed_pco2_ensemble_mean.png"),
                xlab=r"Predicted pCO$_2$ (ensemble mean)",
                ylab=r"Observed pCO$_2$",
                title=f"Reconstructed pCO2-space test panel | {scope_name}"
            )

    # -------------------------
    # Return test predictions for root-level merge
    # -------------------------
    test_pred_df = pd.DataFrame(index=test_df.index)
    if te_pred_mean is not None and len(test_df) > 0:
        target_pred_col = f"{FINAL_TARGET}_pred"
        target_pred_std_col = f"{FINAL_TARGET}_pred_std"

        test_pred_df["target_pred"] = te_pred_mean
        test_pred_df["target_pred_std"] = te_pred_std
        test_pred_df[target_pred_col] = te_pred_mean
        test_pred_df[target_pred_std_col] = te_pred_std
        test_pred_df["pCO2_pred"] = reconstruct_pco2_from_target(test_df, te_pred_mean)
        test_pred_df["pCO2_pred_std"] = te_pred_std
        test_pred_df["Model_scope_used"] = scope_name
        test_pred_df["Zone_model_used"] = str(zone_key) if zone_key is not None else "GLOBAL"
        test_pred_df["Modeling_mode_used"] = MODELING_MODE

        if ENABLE_PRIOR_BASELINE and FINAL_BASELINE_PCO2_NAME in test_df.columns:
            test_pred_df["pCO2_pred_prior_baseline"] = pd.to_numeric(
                test_df[FINAL_BASELINE_PCO2_NAME], errors="coerce"
            )

    for mdl in member_models:
        del mdl
    plt.close("all")
    gc.collect()

    return metrics_rows, fi_summary, test_pred_df


# =========================================================
# 12. Data preparation helpers
# =========================================================
def _load_and_prepare_dataframe():
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"Input CSV not found: {CSV_PATH}")

    print(f"[Input] {CSV_PATH}")
    df = pd.read_csv(CSV_PATH, low_memory=False)

    required_cols = [TIME_COL, MONTH_COL, LAT_COL, LON_COL]
    if ENABLE_ZONE_MODELING:
        required_cols.append(ZONE_COL)

    missing_required = [c for c in required_cols if c not in df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    df = prepare_time_features(df)
    df = prepare_geo_features(df)
    df, resolved_cols = resolve_or_build_core_fields(df)

    df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce")
    df[MONTH_COL] = pd.to_numeric(df[MONTH_COL], errors="coerce")
    df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
    df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")

    if ZONE_COL in df.columns:
        df["_ZoneKey"] = build_zone_key_series(df[ZONE_COL])
    else:
        df["_ZoneKey"] = np.nan

    df = df[(df[TIME_COL] >= YEAR_START) & (df[TIME_COL] <= YEAR_END)].copy()

    essential_cols = [TIME_COL, MONTH_COL, LAT_COL, LON_COL, FINAL_LABEL_NAME, FINAL_TARGET]
    if target_requires_prior_field() and FINAL_PRIOR_NAME in df.columns:
        essential_cols.append(FINAL_PRIOR_NAME)
    if target_requires_prior_clim_field() and FINAL_PRIOR_CLIM_NAME in df.columns:
        essential_cols.append(FINAL_PRIOR_CLIM_NAME)
    if target_requires_delta_t_field() and FINAL_DELTA_T_NAME in df.columns:
        essential_cols.append(FINAL_DELTA_T_NAME)
    if ENABLE_PRIOR_BASELINE and FINAL_BASELINE_PCO2_NAME in df.columns:
        essential_cols.append(FINAL_BASELINE_PCO2_NAME)

    if ENFORCE_VALID_FEATURES:
        before = len(df)
        df = enforce_valid_feature_rows(
            df,
            features=essential_cols,
            sentinel_values=SENTINEL_VALUES,
            feature_ranges=FEATURE_RANGES
        )
        after = len(df)
        print(f"[Prepare] Essential field filter: kept {after}/{before}.")

    df = df[(df[FINAL_LABEL_NAME] > 0) & (df[FINAL_LABEL_NAME] <= 1100)].copy()

    train_df = df[~df[TIME_COL].isin(TEST_YEARS)].copy()
    test_df = df[df[TIME_COL].isin(TEST_YEARS)].copy()

    print(f"[Split] train={len(train_df)} test={len(test_df)}")
    print(f"[Split] TEST_YEARS={TEST_YEARS}")

    return df, train_df, test_df, resolved_cols


def _save_root_summary(train_df, test_df, extra_lines):
    with open(os.path.join(OUTPUT_DIR, "training_summary_root.txt"), "w", encoding="utf-8") as f:
        f.write(f"Mode: {get_modeling_mode_description()}\n")
        f.write(f"Modeling mode: {MODELING_MODE}\n")
        f.write(f"Run mode: {RUN_MODE}\n")
        f.write(f"Final target: {FINAL_TARGET}\n")
        f.write(f"Train samples: {len(train_df)}\n")
        f.write(f"Test samples: {len(test_df)}\n")
        for line in extra_lines:
            f.write(line.rstrip() + "\n")


# =========================================================
# 13. Zone-wise main pipeline
# =========================================================
def process_zone_training():
    _, train_df, test_df, resolved_cols = _load_and_prepare_dataframe()

    zone_keys_train = sort_zone_keys(train_df["_ZoneKey"].dropna().unique().tolist())
    zone_keys_test = sort_zone_keys(test_df["_ZoneKey"].dropna().unique().tolist())

    print(f"[Zonewise] train zones = {zone_keys_train}")
    print(f"[Zonewise] test zones  = {zone_keys_test}")

    combined_test = test_df.copy()
    target_pred_col = f"{FINAL_TARGET}_pred"
    target_pred_std_col = f"{FINAL_TARGET}_pred_std"

    combined_test["target_pred"] = np.nan
    combined_test["target_pred_std"] = np.nan
    combined_test[target_pred_col] = np.nan
    combined_test[target_pred_std_col] = np.nan
    combined_test["pCO2_pred"] = np.nan
    combined_test["pCO2_pred_std"] = np.nan
    combined_test["Model_scope_used"] = pd.Series([None] * len(combined_test), index=combined_test.index, dtype="object")
    combined_test["Zone_model_used"] = pd.Series([None] * len(combined_test), index=combined_test.index, dtype="object")
    combined_test["Modeling_mode_used"] = MODELING_MODE

    if ENABLE_PRIOR_BASELINE and FINAL_BASELINE_PCO2_NAME in combined_test.columns:
        combined_test["pCO2_pred_prior_baseline"] = pd.to_numeric(
            combined_test[FINAL_BASELINE_PCO2_NAME], errors="coerce"
        )

    all_metrics_frames = []
    zone_summary_rows = []
    skipped_rows = []

    for iz, zone_key in enumerate(zone_keys_train, start=1):
        zone_name = f"Zone={zone_key}"
        print("=" * 80)
        print(f"[{iz}/{len(zone_keys_train)}] {zone_name}")
        print("=" * 80)

        train_z = train_df[train_df["_ZoneKey"] == zone_key].copy().reset_index(drop=True)
        test_z = test_df[test_df["_ZoneKey"] == zone_key].copy()

        n_train_z = len(train_z)
        n_test_z = len(test_z)

        if n_train_z < MIN_SAMPLES:
            skipped_rows.append({
                "ZoneKey": zone_key,
                "n_train": n_train_z,
                "n_test": n_test_z,
                "reason": f"train samples < MIN_SAMPLES ({n_train_z} < {MIN_SAMPLES})"
            })
            print(f"[Skip] {zone_name}: train samples < MIN_SAMPLES")
            continue

        train_z["sample_weight"] = build_density_sample_weight(train_z)
        if len(train_z) > 0:
            sw = train_z["sample_weight"].values
            print(
                f"[{zone_name}] sample_weight stats: "
                f"min={np.min(sw):.4f} max={np.max(sw):.4f} mean={np.mean(sw):.4f}"
            )

        chla_offset = compute_chla_offset(
            pd.to_numeric(train_z["Chla"], errors="coerce").to_numpy(dtype=float)
        ) if "Chla" in train_z.columns else 1e-6

        feature_stats = compute_feature_stats(train_z, chla_offset)

        selected_candidates, X_train_all = select_available_features_from_train(
            train_z, chla_offset, feature_stats
        )

        if len(selected_candidates) == 0:
            skipped_rows.append({
                "ZoneKey": zone_key,
                "n_train": n_train_z,
                "n_test": n_test_z,
                "reason": "no usable predictors after screening"
            })
            print(f"[Skip] {zone_name}: no usable predictors after screening")
            continue

        X_train_all = X_train_all.reindex(columns=selected_candidates, fill_value=np.nan)

        zone_output_dir = os.path.join(OUTPUT_DIR, zone_dir_name(zone_key))

        metrics_rows, fi_summary, test_pred_df = run_scope_ensemble_model(
            train_df=train_z,
            test_df=test_z,
            selected_candidates=selected_candidates,
            resolved_cols=resolved_cols,
            chla_offset=chla_offset,
            feature_stats=feature_stats,
            X_tr_all_df=X_train_all,
            output_dir=zone_output_dir,
            scope_name=zone_name,
            zone_key=zone_key
        )

        if metrics_rows:
            mdf = pd.DataFrame(
                metrics_rows,
                columns=["Group", "Split", "MetricSpace", "MSE", "RMSE", "MAE", "R2", "MAPE"]
            )
            mdf.insert(0, "ZoneKey", zone_key)
            mdf.insert(1, "n_train_zone", n_train_z)
            mdf.insert(2, "n_test_zone", n_test_z)
            mdf.insert(3, "ModelingMode", MODELING_MODE)
            all_metrics_frames.append(mdf)

        if len(test_pred_df) > 0:
            for col in test_pred_df.columns:
                combined_test.loc[test_pred_df.index, col] = test_pred_df[col]

        zone_summary_rows.append({
            "ZoneKey": zone_key,
            "n_train": n_train_z,
            "n_test": n_test_z,
            "n_features_after_screening": len(selected_candidates),
            "output_dir": zone_output_dir,
            "status": "trained",
            "ModelingMode": MODELING_MODE
        })

        _hard_memory_cleanup()

    unseen_test_zones = sorted(set(zone_keys_test) - set(zone_keys_train))
    for zone_key in unseen_test_zones:
        n_test_z = int((test_df["_ZoneKey"] == zone_key).sum())
        skipped_rows.append({
            "ZoneKey": zone_key,
            "n_train": 0,
            "n_test": n_test_z,
            "reason": "zone exists in test only; no training samples"
        })

    if all_metrics_frames:
        pd.concat(all_metrics_frames, axis=0, ignore_index=True).to_csv(
            os.path.join(OUTPUT_DIR, "all_metrics_by_zone.csv"), index=False
        )

    pd.DataFrame(zone_summary_rows).to_csv(
        os.path.join(OUTPUT_DIR, "zone_training_summary.csv"), index=False
    )

    if skipped_rows:
        pd.DataFrame(skipped_rows).to_csv(
            os.path.join(OUTPUT_DIR, "zone_skipped_summary.csv"), index=False
        )

    combined_test = combined_test.sort_index().reset_index(drop=True)
    combined_test.to_csv(os.path.join(OUTPUT_DIR, "test_with_pred.csv"), index=False)

    _save_root_summary(
        train_df=train_df,
        test_df=test_df,
        extra_lines=[
            f"Zone column: {ZONE_COL}",
            f"Train zones: {zone_keys_train}",
            f"Test zones: {zone_keys_test}",
            f"Skipped zones: {len(skipped_rows)}",
        ]
    )

    print(f"[Done] Zone-wise outputs in: {OUTPUT_DIR}")


# =========================================================
# 14. Global unified main pipeline (NEW)
# =========================================================
def process_global_training():
    _, train_df, test_df, resolved_cols = _load_and_prepare_dataframe()

    print("[Global] One single global model will be trained.")
    print("[Global] Zone column is ignored during model fitting.")

    train_g = train_df.copy().reset_index(drop=True)
    test_g = test_df.copy()

    if len(train_g) < MIN_SAMPLES:
        raise ValueError(f"[Global] train samples < MIN_SAMPLES ({len(train_g)} < {MIN_SAMPLES})")

    train_g["sample_weight"] = build_density_sample_weight(train_g)
    if len(train_g) > 0:
        sw = train_g["sample_weight"].values
        print(
            f"[Global] sample_weight stats: "
            f"min={np.min(sw):.4f} max={np.max(sw):.4f} mean={np.mean(sw):.4f}"
        )

    chla_offset = compute_chla_offset(
        pd.to_numeric(train_g["Chla"], errors="coerce").to_numpy(dtype=float)
    ) if "Chla" in train_g.columns else 1e-6

    feature_stats = compute_feature_stats(train_g, chla_offset)

    selected_candidates, X_train_all = select_available_features_from_train(
        train_g, chla_offset, feature_stats
    )

    if len(selected_candidates) == 0:
        raise ValueError("[Global] no usable predictors after screening")

    X_train_all = X_train_all.reindex(columns=selected_candidates, fill_value=np.nan)

    metrics_rows, fi_summary, test_pred_df = run_scope_ensemble_model(
        train_df=train_g,
        test_df=test_g,
        selected_candidates=selected_candidates,
        resolved_cols=resolved_cols,
        chla_offset=chla_offset,
        feature_stats=feature_stats,
        X_tr_all_df=X_train_all,
        output_dir=OUTPUT_DIR,
        scope_name="GLOBAL",
        zone_key="GLOBAL"
    )

    if metrics_rows:
        mdf = pd.DataFrame(
            metrics_rows,
            columns=["Group", "Split", "MetricSpace", "MSE", "RMSE", "MAE", "R2", "MAPE"]
        )
        mdf.insert(0, "Scope", "GLOBAL")
        mdf.insert(1, "n_train_scope", len(train_g))
        mdf.insert(2, "n_test_scope", len(test_g))
        mdf.insert(3, "ModelingMode", MODELING_MODE)
        mdf.to_csv(os.path.join(OUTPUT_DIR, "all_metrics_global.csv"), index=False)

    combined_test = test_g.copy()
    target_pred_col = f"{FINAL_TARGET}_pred"
    target_pred_std_col = f"{FINAL_TARGET}_pred_std"

    for col in ["target_pred", "target_pred_std", target_pred_col, target_pred_std_col,
                "pCO2_pred", "pCO2_pred_std", "Model_scope_used", "Zone_model_used", "Modeling_mode_used"]:
        if col not in combined_test.columns:
            combined_test[col] = np.nan

    if ENABLE_PRIOR_BASELINE and FINAL_BASELINE_PCO2_NAME in combined_test.columns:
        combined_test["pCO2_pred_prior_baseline"] = pd.to_numeric(
            combined_test[FINAL_BASELINE_PCO2_NAME], errors="coerce"
        )

    if len(test_pred_df) > 0:
        for col in test_pred_df.columns:
            combined_test.loc[test_pred_df.index, col] = test_pred_df[col]

    combined_test = combined_test.sort_index().reset_index(drop=True)
    combined_test.to_csv(os.path.join(OUTPUT_DIR, "test_with_pred.csv"), index=False)

    global_summary_df = pd.DataFrame([{
        "Scope": "GLOBAL",
        "n_train": len(train_g),
        "n_test": len(test_g),
        "n_features_after_screening": len(selected_candidates),
        "output_dir": OUTPUT_DIR,
        "status": "trained",
        "ModelingMode": MODELING_MODE
    }])
    global_summary_df.to_csv(os.path.join(OUTPUT_DIR, "global_training_summary.csv"), index=False)

    _save_root_summary(
        train_df=train_df,
        test_df=test_df,
        extra_lines=[
            "Zone column ignored during fitting: yes",
            f"Global train samples: {len(train_g)}",
            f"Global test samples: {len(test_g)}",
            f"Selected candidate features before final selection: {len(selected_candidates)}",
        ]
    )

    print(f"[Done] Global outputs in: {OUTPUT_DIR}")


# =========================================================
# 15. Dispatcher
# =========================================================
def process_training():
    if ENABLE_ZONE_MODELING:
        process_zone_training()
    else:
        process_global_training()


# =========================================================
# 16. Hard memory cleanup
# =========================================================
def _hard_memory_cleanup():
    try:
        plt.close("all")
    except Exception:
        pass
    for _ in range(2):
        gc.collect()
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


# =========================================================
# 17. Main
# =========================================================
if __name__ == "__main__":
    print(f"[Config] Input CSV = {CSV_PATH}")
    print(f"[Config] YEAR_START={YEAR_START}, YEAR_END={YEAR_END}")
    print(f"[Config] TEST_YEARS = {TEST_YEARS}")
    print(f"[Config] RUN_MODE = {RUN_MODE}")
    print(f"[Config] FINAL_TARGET = {FINAL_TARGET}")
    print(f"[Config] pCO2 reconstruction mode = {get_reconstruction_mode()}")
    print(f"[Config] Baseline mode = {get_baseline_mode_description()}")
    print(f"[Config] MODELING_MODE = {MODELING_MODE}")
    print(f"[Config] Modeling mode desc = {get_modeling_mode_description()}")
    print(f"[Config] Zone column = {ZONE_COL}")
    print(f"[Config] Prior baseline enabled = {ENABLE_PRIOR_BASELINE}")
    print(f"[Config] Ensemble enabled = {ENABLE_ENSEMBLE}")
    print(f"[Config] Ensemble size = {ENSEMBLE_SIZE}")
    print(f"[Config] Bootstrap rows = {ENSEMBLE_BOOTSTRAP_ROWS}, frac = {ENSEMBLE_BOOTSTRAP_FRAC}")
    print(f"[Config] Feature engineering enabled = {ENABLE_FEATURE_ENGINEERING}")
    print(f"[Config] Feature selection enabled = {ENABLE_FEATURE_SELECTION}")
    print(f"[Config] OUTPUT_DIR = {OUTPUT_DIR}")
    print(f"[Config] DensityWeighting={ENABLE_DENSITY_WEIGHTING} "
          f"(grid={WEIGHT_GRID_RESOLUTION_DEG}°, time={WEIGHT_TIME_RESOLUTION_YR}yr)")
    print("[Config] CV = DecadeBlockKFold (10-year blocks)")

    process_training()

    print("All done.")
    _hard_memory_cleanup()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
绘制 pCO2 分区训练后的特征重要性热图（单层/单子图版）
热图线条/框线样式参考论文附图风格：
- 每个单元格带浅灰细边框
- 0 值/空值显示为统一浅灰底
- 右侧带 mean importance 条形图
- 输出 PNG / PDF / 矩阵 CSV

数据来源：
每个分区目录下的 feature_importance_ensemble.csv
例如：
/data/wang/Result_pCO2/models_ML/current_model/zonewise_model/Zone_1/feature_importance_ensemble.csv
/data/wang/Result_pCO2/models_ML/current_model/zonewise_model/Zone_2/feature_importance_ensemble.csv
...

输出：
- zone_feature_importance_single.png
- zone_feature_importance_single.pdf
- zone_feature_importance_matrix_surface.csv
"""

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize

# =========================================================
# 0. Path configuration
# =========================================================
ZONEWISE_ROOT = "/data/wang/Result_pCO2/models_ML/current_model/zonewise_model"
OUT_DIR = os.path.join(ZONEWISE_ROOT, "feature_importance_plot")
os.makedirs(OUT_DIR, exist_ok=True)

# Enable the shared style file if available; otherwise use the default style.
STYLE_PATH = "/data/wang/Result_Data/Code/Phd/paper_unified.mplstyle"
USE_STYLE = os.path.exists(STYLE_PATH)

# =========================================================
# 1. Plot configuration
# =========================================================
# Whether to normalize each column to sum to 1, closer to relative importance.
NORMALIZE_BY_ZONE = True

# Whether the mean importance bars on the right ignore zeros.
EXCLUDE_ZERO_IN_MEAN = True

# Whether to annotate values in the heatmap.
ANNOTATE = True

# Show only the top N features; if None, show all features.
TOP_N_FEATURES = None

# Provide a list to force a display order; set to None to keep the automatic order.
FEATURE_ORDER = [
    "SST","SST_anom","CO2","CO2atm","log10_Chla", "SSS", "SSS_anom",
    "U","U_anom", "V", "V_anom", "SLA_ERA5","MLD", "EKE_ERA5",
    "Siconc",
    "geo_nx", "geo_ny", "geo_nz",
    "NINO34",
]

# Zone display-name mapping; adjust according to the actual zone IDs.
ZONE_LABEL_MAP = {
    "1": "ARC",
    "2": "PAC",
    "3": "ATL",
    "4": "IND",
    "5": "SO",
}

# Colormap
CMAP_NAME = "inferno"

# Font sizes
TITLE_FONTSIZE = 15
AXIS_LABEL_FONTSIZE = 13
TICK_FONTSIZE = 11
ANNOTATE_FONTSIZE = 9

# Background color for zero or missing cells
ZERO_FACE = "#F0F0F0"

# Heatmap cell border style, following the reference figure.
CELL_EDGE_COLOR = "#D9D9D9"
CELL_EDGE_WIDTH = 0.8

# =========================================================
# 2. Utility functions
# =========================================================
def extract_zone_key_from_dirname(dirname: str) -> str:
    """
    从 Zone_1 / Zone_2 / Zone_3.0 中提取分区编号字符串
    """
    base = os.path.basename(dirname.rstrip("/"))
    m = re.match(r"Zone_(.+)", base)
    if m:
        return m.group(1)
    return base


def sort_zone_keys(keys):
    def _k(z):
        try:
            return (0, float(z))
        except Exception:
            return (1, str(z))
    return sorted(keys, key=_k)


def nonzero_mean_along_axis1(arr: np.ndarray, exclude_zero=True) -> np.ndarray:
    arr = np.asarray(arr, float)
    if exclude_zero:
        mask = np.isfinite(arr) & (arr > 0)
        sums = np.where(mask, arr, 0.0).sum(axis=1)
        cnts = mask.sum(axis=1)
    else:
        mask = np.isfinite(arr)
        sums = np.where(mask, arr, 0.0).sum(axis=1)
        cnts = mask.sum(axis=1)

    means = np.divide(sums, cnts, out=np.zeros(arr.shape[0], dtype=float), where=cnts > 0)
    return means


def pick_feature_column(df: pd.DataFrame) -> str:
    """
    自动识别特征列名
    """
    for c in ["Feature", "feature", "features"]:
        if c in df.columns:
            return c
    raise KeyError("未找到特征名列，期望列名之一：Feature / feature / features")


def pick_importance_column(df: pd.DataFrame) -> str:
    """
    自动识别重要性列名
    """
    for c in ["Importance_mean", "importance_mean", "Importance", "importance"]:
        if c in df.columns:
            return c
    raise KeyError("未找到重要性列，期望列名之一：Importance_mean / importance_mean / Importance / importance")


def prettify_feature_name(name: str) -> str:
    """
    将字段名改成更适合画图的显示名
    """
    rename_map = {
        "CO2": "$x$CO2",
        "NINO34": "NINO3.4_anom",
        "SLA_ERA5": "SLA",
        "EKE_ERA5": "EKE",
        "log10_Chla": "log10(Chla)",
        "U": "U10",
        "V": "V10",
        "geo_nz":"lat_sin",
        
    }
    return rename_map.get(name, name)


# =========================================================
# 3. Read feature importance for each zone
# =========================================================
zone_dirs = sorted(glob.glob(os.path.join(ZONEWISE_ROOT, "Zone_*")))
if len(zone_dirs) == 0:
    raise FileNotFoundError(f"未找到任何分区目录：{ZONEWISE_ROOT}/Zone_*")

zone_series = {}
for zdir in zone_dirs:
    csv_path = os.path.join(zdir, "feature_importance_ensemble.csv")
    if not os.path.exists(csv_path):
        print(f"[跳过] 缺少文件: {csv_path}")
        continue

    zone_key = extract_zone_key_from_dirname(zdir)
    df = pd.read_csv(csv_path)

    feat_col = pick_feature_column(df)
    imp_col = pick_importance_column(df)

    s = pd.Series(
        pd.to_numeric(df[imp_col], errors="coerce").values,
        index=df[feat_col].astype(str).values,
        name=zone_key
    )
    s = s.groupby(level=0).mean()  # 防止重复特征名
    zone_series[zone_key] = s

if len(zone_series) == 0:
    raise RuntimeError("没有成功读到任何 feature_importance_ensemble.csv")

# Assemble the Feature x Zone matrix
zone_keys = sort_zone_keys(list(zone_series.keys()))
mat = pd.concat([zone_series[z] for z in zone_keys], axis=1)
mat.columns = zone_keys
mat = mat.fillna(0.0)

# =========================================================
# 4. Optional column normalization
# =========================================================
if NORMALIZE_BY_ZONE:
    colsum = mat.sum(axis=0).replace(0.0, np.nan)
    mat = mat.div(colsum, axis=1).fillna(0.0)

# =========================================================
# 5. Feature sorting
# =========================================================
mean_imp = pd.Series(
    nonzero_mean_along_axis1(mat.values, exclude_zero=EXCLUDE_ZERO_IN_MEAN),
    index=mat.index
).sort_values(ascending=False)

if FEATURE_ORDER is not None:
    ordered = [f for f in FEATURE_ORDER if f in mat.index]
    remaining = [f for f in mean_imp.index if f not in ordered]
    final_features = ordered + remaining
else:
    final_features = list(mean_imp.index)

if TOP_N_FEATURES is not None:
    final_features = final_features[:TOP_N_FEATURES]

mat = mat.reindex(final_features)

display_index = [prettify_feature_name(f) for f in mat.index]
display_columns = [ZONE_LABEL_MAP.get(z, z) for z in mat.columns]

# =========================================================
# 6. Plotting
# =========================================================
if USE_STYLE:
    plt.style.use(STYLE_PATH)
else:
    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.linewidth": 1.0,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "savefig.dpi": 350,
    })

fig = plt.figure(figsize=(11.5, 8.2), facecolor="white")
gs = fig.add_gridspec(
    nrows=1, ncols=2,
    width_ratios=[8.5, 1.8],
    wspace=0.08
)

ax_hm = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1])

arr = mat.values.astype(float)
n_rows, n_cols = arr.shape

# ---------------------------------------------------------
# Heatmap body: pcolormesh with light-gray cell borders
# ---------------------------------------------------------
vmax = float(np.nanmax(arr)) if np.isfinite(arr).any() else 1.0
if vmax <= 0:
    vmax = 1.0
norm = Normalize(vmin=0.0, vmax=vmax)

# Copy the colormap and set the color for masked values
cmap = plt.get_cmap(CMAP_NAME).copy()
cmap.set_bad(color=ZERO_FACE)

# Display zero values as blank gray cells
arr_plot = arr.copy()
arr_plot[np.isclose(arr_plot, 0.0)] = np.nan
arr_masked = np.ma.masked_invalid(arr_plot)

# Grid boundaries
x = np.arange(n_cols + 1)
y = np.arange(n_rows + 1)

# Draw the gridded heatmap with pcolormesh
im = ax_hm.pcolormesh(
    x, y, arr_masked,
    cmap=cmap,
    norm=norm,
    shading="flat",
    edgecolors=CELL_EDGE_COLOR,
    linewidth=CELL_EDGE_WIDTH,
    antialiased=False
)

# Keep the first row at the top
ax_hm.set_xlim(0, n_cols)
ax_hm.set_ylim(n_rows, 0)

# Center the tick labels
ax_hm.set_xticks(np.arange(n_cols) + 0.5)
ax_hm.set_xticklabels(display_columns, rotation=0, fontsize=TICK_FONTSIZE)
ax_hm.set_yticks(np.arange(n_rows) + 0.5)
ax_hm.set_yticklabels(display_index, fontsize=TICK_FONTSIZE)

ax_hm.set_xlabel("Ocean basin", fontsize=AXIS_LABEL_FONTSIZE)
ax_hm.set_ylabel("Feature", fontsize=AXIS_LABEL_FONTSIZE)

# Background color
ax_hm.set_facecolor(ZERO_FACE)

# Disable extra grid lines to avoid conflicts with cell borders
ax_hm.grid(False)

# Value annotations
if ANNOTATE:
    for i in range(n_rows):
        for j in range(n_cols):
            v = arr[i, j]
            if np.isclose(v, 0.0) or not np.isfinite(v):
                continue
            txt = f"{v:.2f}"

            rgba = cmap(norm(v))
            luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
            color = "white" if luminance < 0.5 else "black"

            ax_hm.text(
                j + 0.5, i + 0.5, txt,
                ha="center", va="center",
                fontsize=ANNOTATE_FONTSIZE,
                color=color
            )

for spine in ax_hm.spines.values():
    spine.set_visible(False)

# ---------------------------------------------------------
# Mean-importance bar chart on the right
# ---------------------------------------------------------
row_mean = nonzero_mean_along_axis1(arr, exclude_zero=EXCLUDE_ZERO_IN_MEAN)
ypos = np.arange(n_rows)

bar_colors = [cmap(norm(v)) if v > 0 else ZERO_FACE for v in row_mean]

ax_bar.barh(
    ypos + 0.5, row_mean,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.8,
    height=0.78
)

ax_bar.set_ylim(ax_hm.get_ylim())
ax_bar.set_yticks(np.arange(n_rows) + 0.5)
ax_bar.set_yticklabels([])
ax_bar.set_xlabel("Mean importance", fontsize=AXIS_LABEL_FONTSIZE)
ax_bar.tick_params(axis="x", labelsize=TICK_FONTSIZE)
ax_bar.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.35)

xmax = max(0.4, float(np.nanmax(row_mean)) * 1.10 if np.isfinite(np.nanmax(row_mean)) else 0.4)
ax_bar.set_xlim(0, xmax)

for spine in ax_bar.spines.values():
    spine.set_linewidth(0.9)
    spine.set_color("black")

# ---------------------------------------------------------
# Shared colorbar
# ---------------------------------------------------------
cbar = fig.colorbar(im, ax=[ax_hm, ax_bar], fraction=0.03, pad=0.03)
cbar.set_label(
    "Relative importance" if NORMALIZE_BY_ZONE else "Mean feature importance",
    fontsize=AXIS_LABEL_FONTSIZE
)
cbar.ax.tick_params(labelsize=TICK_FONTSIZE)

# =========================================================
# 7. Save outputs
# =========================================================
mat_out = mat.copy()
mat_out.index.name = "Feature"
mat_out.columns = display_columns
csv_out = os.path.join(OUT_DIR, "zone_feature_importance_matrix_surface.csv")
mat_out.to_csv(csv_out)

png_path = os.path.join(OUT_DIR, "zone_feature_importance_single.png")
pdf_path = os.path.join(OUT_DIR, "zone_feature_importance_single.pdf")

plt.savefig(png_path, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(f"[完成] 矩阵保存: {csv_out}")
print(f"[完成] PNG: {png_path}")
print(f"[完成] PDF: {pdf_path}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
plot_prior_train_test_scatter_from_saved_models.py
==================================================

功能
----
基于“已经训练完成”的模型输出目录，重新加载保存的 ensemble member 模型，
对训练集与测试集分别做推理，并绘制 1 行 3 列论文级散点图：

    (a) Prior baseline (train + test combined)
    (b) Train scatter
    (c) Test scatter

修改说明
--------
1) 不显示总题名；
2) 不显示每个子图题名；
3) 使用 (a)、(b)、(c) 作为子图编号；
4) 其他绘图样式、指标统计和输出逻辑保持不变。
"""

from __future__ import annotations

import os
import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")


# =========================================================
# 0. User configuration
# =========================================================
MODEL_OUTPUT_DIR = "/data/wang/Result_pCO2/models_ML/current_model/zonewise_model"
CSV_PATH = "/data/wang/Result_pCO2/allpco2/SOCAT_TRAIN.csv"
OUTPUT_DIR = os.path.join(MODEL_OUTPUT_DIR, "scatter_rebuild")

TIME_COL = "Year"
MONTH_COL = "Month"
LAT_COL = "Latitude"
LON_COL = "Longitude"
ZONE_COL = "Zone"

YEAR_START = 1982
YEAR_END = 2024
TEST_YEARS = [1984, 1988, 1995, 2004, 2013, 2021]

DATE_CANDIDATES = ["Date"]
LABEL_CANDIDATES = ["pCO2"]
PRIOR_CANDIDATES = ["pCO2_prior"]
PRIOR_CLIM_CANDIDATES = [
    "pCO2_prior_clim_mapped", "pco2_prior_clim_mapped",
    "pCO2_prior_clim", "pco2_prior_clim"
]
R_STAR_T_CANDIDATES = ["r_star_T"]
DELTA_PCO2_T_CANDIDATES = ["delta_pCO2_T"]

FINAL_PRIOR_NAME = "pCO2_prior_used"
FINAL_PRIOR_CLIM_NAME = "pCO2_prior_clim_used"
FINAL_LABEL_NAME = "pCO2_label_used"
FINAL_DELTA_T_NAME = "delta_pCO2_T_used"
FINAL_RECON_OFFSET_NAME = "pCO2_reconstruction_offset_used"
FINAL_BASELINE_PCO2_NAME = "pCO2_baseline_used"

DPI = 450
FIGSIZE = (18.8, 5.8)

POINT_SIZE = 5
POINT_ALPHA = 0.28
HEXBIN_GRIDSIZE = 120

ANNOT_FONTSIZE = 10.2
LABEL_FONTSIZE = 12
PANEL_LABEL_FONTSIZE = 14
TICK_FONTSIZE = 10.5
CBAR_FONTSIZE = 10

SPINE_LW = 1.1
ONE2ONE_COLOR = "0.35"
ONE2ONE_LW = 1.0

USE_HEXBIN = True


# =========================================================
# 1. Matplotlib style
# =========================================================
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "STIXGeneral"],
    "mathtext.fontset": "stix",
    "axes.linewidth": SPINE_LW,
    "axes.labelsize": LABEL_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2.5,
    "ytick.minor.size": 2.5,
    "legend.frameon": False,
    "savefig.dpi": DPI,
    "figure.dpi": DPI,
})


# =========================================================
# 2. General functions
# =========================================================
def find_first_existing(columns, candidates, required=False, field_desc="field"):
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise KeyError(f"Missing required {field_desc}. Candidates: {candidates}")
    return None


def normalize_zone_value(v):
    if pd.isna(v):
        return np.nan
    try:
        fv = float(v)
        if not np.isfinite(fv):
            return np.nan
        if abs(fv - round(fv)) < 1e-9:
            return str(int(round(fv)))
        return f"{fv:g}"
    except Exception:
        s = str(v).strip()
        return s if s != "" else np.nan


def build_zone_key_series(series):
    return series.apply(normalize_zone_value)


def sort_zone_keys(zone_keys):
    def _k(z):
        try:
            return (0, float(z))
        except Exception:
            return (1, str(z))
    return sorted(zone_keys, key=_k)


def safe_mape(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    m = np.isfinite(y_true) & np.isfinite(y_pred) & (np.abs(y_true) > eps)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs((y_pred[m] - y_true[m]) / y_true[m])) * 100.0)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[m]
    y_pred = y_pred[m]

    if y_true.size == 0:
        return {
            "MSE": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "MAPE": np.nan,
            "Bias": np.nan,
            "n": 0
        }

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    mape = safe_mape(y_true, y_pred)
    bias = float(np.mean(y_pred - y_true))

    return {
        "MSE": float(mse),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "MAPE": mape,
        "Bias": bias,
        "n": int(y_true.size)
    }


def make_axis_limits(*arrays, pad_frac=0.03):
    vals = []
    for arr in arrays:
        a = np.asarray(arr, float).ravel()
        a = a[np.isfinite(a)]
        if a.size > 0:
            vals.append(a)

    if len(vals) == 0:
        return 0.0, 1.0

    vals = np.concatenate(vals)
    vmin = float(np.nanmin(vals))
    vmax = float(np.nanmax(vals))

    if vmax <= vmin:
        vmax = vmin + 1.0

    pad = pad_frac * (vmax - vmin)
    return vmin - pad, vmax + pad


# =========================================================
# 3. Data preparation consistent with the training script
# =========================================================
def prepare_time_features(df):
    out = df.copy()
    out[TIME_COL] = pd.to_numeric(out[TIME_COL], errors="coerce")
    out[MONTH_COL] = pd.to_numeric(out[MONTH_COL], errors="coerce")

    if "month_sin" not in out.columns:
        out["month_sin"] = np.sin(2.0 * np.pi * (out[MONTH_COL] - 1.0) / 12.0)
    if "month_cos" not in out.columns:
        out["month_cos"] = np.cos(2.0 * np.pi * (out[MONTH_COL] - 1.0) / 12.0)

    if "year_norm" not in out.columns:
        y = pd.to_numeric(out[TIME_COL], errors="coerce").to_numpy(dtype=float)
        out["year_norm"] = (y - YEAR_START) / max(1.0, (YEAR_END - YEAR_START))

    if "months_since_1950" not in out.columns:
        yy = pd.to_numeric(out[TIME_COL], errors="coerce").to_numpy(dtype=float)
        mm = pd.to_numeric(out[MONTH_COL], errors="coerce").to_numpy(dtype=float)
        out["months_since_1950"] = (yy - 1950.0) * 12.0 + (mm - 1.0)

    return out


def prepare_geo_features(df):
    out = df.copy()

    lat = pd.to_numeric(out[LAT_COL], errors="coerce").to_numpy(dtype=float, copy=False)
    lon = pd.to_numeric(out[LON_COL], errors="coerce").to_numpy(dtype=float, copy=False)

    lon_m180_180 = ((lon + 180.0) % 360.0) - 180.0
    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon_m180_180)

    if "geo_nx" not in out.columns:
        out["geo_nx"] = np.cos(lat_rad) * np.cos(lon_rad)
    if "geo_ny" not in out.columns:
        out["geo_ny"] = np.cos(lat_rad) * np.sin(lon_rad)
    if "geo_nz" not in out.columns:
        out["geo_nz"] = np.sin(lat_rad)

    return out


def resolve_or_build_core_fields(df, final_target):
    out = df.copy()

    label_col = find_first_existing(
        out.columns, LABEL_CANDIDATES, required=True,
        field_desc="pCO2 label field"
    )
    prior_col = find_first_existing(
        out.columns, PRIOR_CANDIDATES, required=False,
        field_desc="pCO2_prior field"
    )
    prior_clim_col = find_first_existing(
        out.columns, PRIOR_CLIM_CANDIDATES, required=False,
        field_desc="pCO2 prior climatology field"
    )
    r_star_t_col = find_first_existing(
        out.columns, R_STAR_T_CANDIDATES, required=False,
        field_desc="r_star_T field"
    )
    delta_t_col = find_first_existing(
        out.columns, DELTA_PCO2_T_CANDIDATES, required=False,
        field_desc="delta_pCO2_T field"
    )
    date_col = find_first_existing(
        out.columns, DATE_CANDIDATES, required=False,
        field_desc="date field"
    )

    out[FINAL_LABEL_NAME] = pd.to_numeric(out[label_col], errors="coerce")
    out[FINAL_PRIOR_NAME] = (
        pd.to_numeric(out[prior_col], errors="coerce")
        if prior_col is not None else np.nan
    )
    out[FINAL_PRIOR_CLIM_NAME] = (
        pd.to_numeric(out[prior_clim_col], errors="coerce")
        if prior_clim_col is not None else np.nan
    )
    out[FINAL_DELTA_T_NAME] = (
        pd.to_numeric(out[delta_t_col], errors="coerce")
        if delta_t_col is not None else np.nan
    )

    if final_target == "r_star_T":
        if prior_clim_col is None or delta_t_col is None:
            raise KeyError(
                "FINAL_TARGET='r_star_T' requires pCO2_prior_clim + delta_pCO2_T."
            )

        thermal_prior = out[FINAL_PRIOR_CLIM_NAME] + out[FINAL_DELTA_T_NAME]

        if r_star_t_col is not None:
            out["r_star_T"] = pd.to_numeric(out[r_star_t_col], errors="coerce")
        else:
            out["r_star_T"] = out[FINAL_LABEL_NAME] - thermal_prior

        out[FINAL_RECON_OFFSET_NAME] = thermal_prior
        out[FINAL_BASELINE_PCO2_NAME] = thermal_prior

    elif final_target == "pCO2":
        out["pCO2"] = out[FINAL_LABEL_NAME]
        out[FINAL_RECON_OFFSET_NAME] = 0.0
        out[FINAL_BASELINE_PCO2_NAME] = (
            out[FINAL_PRIOR_NAME] if prior_col is not None else np.nan
        )

    else:
        raise ValueError(f"Unsupported FINAL_TARGET={final_target}")

    if date_col is not None:
        out["_Date_used"] = pd.to_datetime(out[date_col], errors="coerce")
    else:
        yy = pd.to_numeric(out[TIME_COL], errors="coerce")
        mm = pd.to_numeric(out[MONTH_COL], errors="coerce")
        out["_Date_used"] = pd.to_datetime(
            dict(year=yy, month=mm, day=15),
            errors="coerce"
        )

    return out


def compute_chla_offset(x):
    x = np.asarray(x, float)
    pos = x[np.isfinite(x) & (x > 0)]

    if pos.size == 0:
        return 1e-6

    p5 = np.nanpercentile(pos, 5)
    return max(1e-6, 0.01 * p5)


def get_log10_chla(df, chla_offset=None):
    if "log10_Chla" in df.columns:
        return pd.to_numeric(df["log10_Chla"], errors="coerce")

    if "Chla" not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)

    z = pd.to_numeric(df["Chla"], errors="coerce").to_numpy(dtype=float, copy=False)

    if chla_offset is None:
        chla_offset = compute_chla_offset(z)

    out = np.where(
        np.isfinite(z),
        np.log10(np.maximum(z, 0.0) + float(chla_offset)),
        np.nan
    )

    return pd.Series(out, index=df.index, dtype=float)


def transform_features(df_part, feature_cols, chla_offset, feature_stats):
    X = pd.DataFrame(index=df_part.index)

    sst = (
        pd.to_numeric(df_part["SST"], errors="coerce")
        if "SST" in df_part.columns
        else pd.Series(np.nan, index=df_part.index)
    )
    sss = (
        pd.to_numeric(df_part["SSS"], errors="coerce")
        if "SSS" in df_part.columns
        else pd.Series(np.nan, index=df_part.index)
    )
    do = (
        pd.to_numeric(df_part["DO"], errors="coerce")
        if "DO" in df_part.columns
        else pd.Series(np.nan, index=df_part.index)
    )
    sst_anom = (
        pd.to_numeric(df_part["SST_anom"], errors="coerce")
        if "SST_anom" in df_part.columns
        else pd.Series(np.nan, index=df_part.index)
    )
    do_anom = (
        pd.to_numeric(df_part["DO_anom"], errors="coerce")
        if "DO_anom" in df_part.columns
        else pd.Series(np.nan, index=df_part.index)
    )
    nino34 = (
        pd.to_numeric(df_part["NINO34"], errors="coerce")
        if "NINO34" in df_part.columns
        else pd.Series(np.nan, index=df_part.index)
    )
    log10_chla = get_log10_chla(df_part, chla_offset)

    sst_c = sst - float(feature_stats.get("SST_mean", np.nan))
    sss_c = sss - float(feature_stats.get("SSS_mean", np.nan))
    do_c = do - float(feature_stats.get("DO_mean", np.nan))
    chla_c = log10_chla - float(feature_stats.get("log10_Chla_mean", np.nan))

    for c in feature_cols:
        if c == "log10_Chla":
            X[c] = log10_chla
        elif c == "SSTc_x_DOc":
            X[c] = sst_c * do_c
        elif c == "SSTc_x_SSSc":
            X[c] = sst_c * sss_c
        elif c == "SSTanom_x_DOanom":
            X[c] = sst_anom * do_anom
        elif c == "SSTanom_x_NINO34":
            X[c] = sst_anom * nino34
        elif c == "DOc_x_log10Chla":
            X[c] = do_c * chla_c
        else:
            if c in df_part.columns:
                X[c] = pd.to_numeric(df_part[c], errors="coerce")
            else:
                X[c] = np.nan

    return X


def reconstruct_pco2_from_target(df_part, target_pred, final_target):
    target_pred = np.asarray(target_pred, float)

    if final_target == "r_star_T":
        offset = pd.to_numeric(
            df_part[FINAL_RECON_OFFSET_NAME],
            errors="coerce"
        ).to_numpy(dtype=float, copy=False)

        return offset + target_pred

    if final_target == "pCO2":
        return target_pred.copy()

    raise ValueError(f"Unsupported FINAL_TARGET={final_target}")


# =========================================================
# 4. Model and metadata loading
# =========================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_member_models(member_dir):
    model_paths = sorted(glob.glob(os.path.join(member_dir, "member_*.cbm")))

    if len(model_paths) == 0:
        raise FileNotFoundError(f"No member_*.cbm found in: {member_dir}")

    models = []
    for mp in model_paths:
        mdl = CatBoostRegressor()
        mdl.load_model(mp)
        models.append(mdl)

    return models, model_paths


def collect_member_predictions(models, X):
    preds = []

    for mdl in models:
        preds.append(np.asarray(mdl.predict(X), float))

    if len(preds) == 0:
        return np.empty((len(X), 0), dtype=float)

    return np.column_stack(preds)


def infer_modeling_mode(root_dir):
    root_meta = os.path.join(root_dir, "metadata.json")

    if os.path.exists(root_meta):
        meta = load_json(root_meta)
        return meta.get("modeling_mode", "global"), {"GLOBAL": root_dir}

    zone_dirs = []
    for p in sorted(Path(root_dir).glob("Zone_*")):
        if p.is_dir() and (p / "metadata.json").exists():
            zone_dirs.append(str(p))

    if len(zone_dirs) == 0:
        raise FileNotFoundError(
            f"Cannot find metadata.json or Zone_*/metadata.json under: {root_dir}"
        )

    mapping = {}
    for zd in zone_dirs:
        m = load_json(os.path.join(zd, "metadata.json"))
        zone_key = m.get("zone_key", None)

        if zone_key is None:
            name = os.path.basename(zd)
            zone_key = name.replace("Zone_", "")

        mapping[str(zone_key)] = zd

    return "zonewise", mapping


# =========================================================
# 5. Data reconstruction and prediction
# =========================================================
def load_and_prepare_dataframe(final_target):
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"Input CSV not found: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH, low_memory=False)

    for c in [TIME_COL, MONTH_COL, LAT_COL, LON_COL]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")

    df = prepare_time_features(df)
    df = prepare_geo_features(df)
    df = resolve_or_build_core_fields(df, final_target)

    df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce")
    df[MONTH_COL] = pd.to_numeric(df[MONTH_COL], errors="coerce")
    df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
    df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")

    if ZONE_COL in df.columns:
        df["_ZoneKey"] = build_zone_key_series(df[ZONE_COL])
    else:
        df["_ZoneKey"] = np.nan

    df = df[(df[TIME_COL] >= YEAR_START) & (df[TIME_COL] <= YEAR_END)].copy()
    df = df[(df[FINAL_LABEL_NAME] > 0) & (df[FINAL_LABEL_NAME] <= 1100)].copy()

    train_df = df[~df[TIME_COL].isin(TEST_YEARS)].copy()
    test_df = df[df[TIME_COL].isin(TEST_YEARS)].copy()

    return df, train_df, test_df


def predict_one_scope(df_part, metadata, models, final_target, scope_name):
    feature_cols = metadata["features"]
    chla_offset = metadata.get("chla_offset", 1e-6)
    feature_stats = metadata.get("feature_stats", {})

    X_df = transform_features(df_part, feature_cols, chla_offset, feature_stats)
    X_df = X_df.reindex(columns=feature_cols, fill_value=np.nan)
    X = X_df.to_numpy(dtype=float)

    member_pred = collect_member_predictions(models, X)

    if member_pred.size:
        pred_mean = np.nanmean(member_pred, axis=1)
        pred_std = np.nanstd(member_pred, axis=1, ddof=0)
    else:
        pred_mean = np.full(len(X), np.nan)
        pred_std = np.full(len(X), np.nan)

    out = df_part.copy()
    out["target_pred"] = pred_mean
    out["target_pred_std"] = pred_std
    out[f"{final_target}_pred"] = pred_mean
    out[f"{final_target}_pred_std"] = pred_std
    out["pCO2_pred"] = reconstruct_pco2_from_target(out, pred_mean, final_target)
    out["pCO2_pred_std"] = pred_std
    out["pCO2_pred_prior_baseline"] = pd.to_numeric(
        out[FINAL_BASELINE_PCO2_NAME],
        errors="coerce"
    )
    out["Model_scope_used"] = scope_name

    return out


def rebuild_predictions_global(root_dir):
    meta = load_json(os.path.join(root_dir, "metadata.json"))
    final_target = meta["final_target"]

    _, train_df, test_df = load_and_prepare_dataframe(final_target)

    models, _ = load_member_models(os.path.join(root_dir, "ensemble_members"))

    train_out = predict_one_scope(train_df, meta, models, final_target, "GLOBAL")
    test_out = predict_one_scope(test_df, meta, models, final_target, "GLOBAL")

    train_out["Zone_model_used"] = "GLOBAL"
    test_out["Zone_model_used"] = "GLOBAL"
    train_out["Modeling_mode_used"] = "global"
    test_out["Modeling_mode_used"] = "global"

    return train_out, test_out, meta


def rebuild_predictions_zonewise(root_dir, zone_dir_map):
    any_zone_dir = next(iter(zone_dir_map.values()))
    any_meta = load_json(os.path.join(any_zone_dir, "metadata.json"))
    final_target = any_meta["final_target"]

    _, train_df, test_df = load_and_prepare_dataframe(final_target)

    train_parts = []
    test_parts = []

    zone_keys_train = sort_zone_keys(train_df["_ZoneKey"].dropna().unique().tolist())
    zone_keys_test = sort_zone_keys(test_df["_ZoneKey"].dropna().unique().tolist())

    for zk in zone_keys_train:
        if str(zk) not in zone_dir_map:
            continue

        zone_dir = zone_dir_map[str(zk)]
        meta = load_json(os.path.join(zone_dir, "metadata.json"))
        models, _ = load_member_models(os.path.join(zone_dir, "ensemble_members"))

        train_z = train_df[train_df["_ZoneKey"] == zk].copy()

        if len(train_z) > 0:
            out_z = predict_one_scope(
                train_z, meta, models, final_target, f"Zone={zk}"
            )
            out_z["Zone_model_used"] = str(zk)
            out_z["Modeling_mode_used"] = "zonewise"
            train_parts.append(out_z)

    for zk in zone_keys_test:
        if str(zk) not in zone_dir_map:
            continue

        zone_dir = zone_dir_map[str(zk)]
        meta = load_json(os.path.join(zone_dir, "metadata.json"))
        models, _ = load_member_models(os.path.join(zone_dir, "ensemble_members"))

        test_z = test_df[test_df["_ZoneKey"] == zk].copy()

        if len(test_z) > 0:
            out_z = predict_one_scope(
                test_z, meta, models, final_target, f"Zone={zk}"
            )
            out_z["Zone_model_used"] = str(zk)
            out_z["Modeling_mode_used"] = "zonewise"
            test_parts.append(out_z)

    train_out = (
        pd.concat(train_parts, axis=0).sort_index()
        if train_parts else train_df.iloc[:0].copy()
    )
    test_out = (
        pd.concat(test_parts, axis=0).sort_index()
        if test_parts else test_df.iloc[:0].copy()
    )

    return train_out, test_out, any_meta


# =========================================================
# 6. Plotting
# =========================================================
def add_panel(ax, y_true, y_pred, panel_label, vmin, vmax, xlab, ylab):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[m]
    y_pred = y_pred[m]

    metrics = compute_metrics(y_true, y_pred)

    if USE_HEXBIN:
        hb = ax.hexbin(
            y_pred,
            y_true,
            gridsize=HEXBIN_GRIDSIZE,
            mincnt=1,
            linewidths=0
        )

        arr = hb.get_array()
        if arr is not None and arr.size > 0:
            vmax_hb = float(np.nanpercentile(arr, 99.0))
            hb.set_clim(0, vmax_hb)

        cbar = plt.colorbar(hb, ax=ax, pad=0.015)
        cbar.set_label("Counts", fontsize=CBAR_FONTSIZE)
        cbar.ax.tick_params(labelsize=CBAR_FONTSIZE)

    else:
        ax.scatter(
            y_pred,
            y_true,
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            edgecolor="none"
        )

    # 1:1 reference line
    ax.plot(
        [vmin, vmax],
        [vmin, vmax],
        "--",
        color=ONE2ONE_COLOR,
        lw=ONE2ONE_LW
    )

    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)

    # Do not show subplot titles; show panel labels only
    ax.text(
        0.00,
        1.035,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=PANEL_LABEL_FONTSIZE,
        fontweight="bold",
        clip_on=False
    )

    ax.grid(alpha=0.20, lw=0.5)

    for spine in ax.spines.values():
        spine.set_linewidth(SPINE_LW)

    txt = (
        f"$R^2$ = {metrics['R2']:.3f}\n"
        f"RMSE = {metrics['RMSE']:.3f}\n"
        f"MAE = {metrics['MAE']:.3f}\n"
        f"MAPE = {metrics['MAPE']:.2f}%\n"
        f"Bias = {metrics['Bias']:.3f}\n"
        f"n = {metrics['n']}"
    )

    ax.text(
        0.03,
        0.97,
        txt,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=ANNOT_FONTSIZE,
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor="white",
            edgecolor="0.6",
            linewidth=0.8,
            alpha=0.96
        )
    )

    return metrics


def plot_prior_train_test_scatter(train_df, test_df, output_dir, modeling_mode, final_target):
    # ---------- prior baseline: combined train and test sets ----------
    y_true_base_train = pd.to_numeric(
        train_df[FINAL_LABEL_NAME],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_pred_base_train = pd.to_numeric(
        train_df["pCO2_pred_prior_baseline"],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_true_base_test = pd.to_numeric(
        test_df[FINAL_LABEL_NAME],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_pred_base_test = pd.to_numeric(
        test_df["pCO2_pred_prior_baseline"],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_true_base = np.concatenate([y_true_base_train, y_true_base_test])
    y_pred_base = np.concatenate([y_pred_base_train, y_pred_base_test])

    # ---------- train / test ----------
    y_true_tr = pd.to_numeric(
        train_df[FINAL_LABEL_NAME],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_pred_tr = pd.to_numeric(
        train_df["pCO2_pred"],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_true_te = pd.to_numeric(
        test_df[FINAL_LABEL_NAME],
        errors="coerce"
    ).to_numpy(dtype=float)

    y_pred_te = pd.to_numeric(
        test_df["pCO2_pred"],
        errors="coerce"
    ).to_numpy(dtype=float)

    # Use a unified coordinate range
    vmin, vmax = make_axis_limits(
        y_true_base, y_pred_base,
        y_true_tr, y_pred_tr,
        y_true_te, y_pred_te
    )

    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE)

    met_base = add_panel(
        axes[0],
        y_true_base,
        y_pred_base,
        panel_label="(a)",
        vmin=vmin,
        vmax=vmax,
        xlab=r"Prior-only predicted pCO$_2$",
        ylab=r"Observed pCO$_2$"
    )

    met_tr = add_panel(
        axes[1],
        y_true_tr,
        y_pred_tr,
        panel_label="(b)",
        vmin=vmin,
        vmax=vmax,
        xlab=r"Predicted pCO$_2$",
        ylab=r"Observed pCO$_2$"
    )

    met_te = add_panel(
        axes[2],
        y_true_te,
        y_pred_te,
        panel_label="(c)",
        vmin=vmin,
        vmax=vmax,
        xlab=r"Predicted pCO$_2$",
        ylab=r"Observed pCO$_2$"
    )

    # Do not show the overall title

    fig.tight_layout(rect=[0, 0, 1, 0.98])

    png_path = os.path.join(output_dir, "prior_train_test_scatter_paper.png")
    pdf_path = os.path.join(output_dir, "prior_train_test_scatter_paper.pdf")

    fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    metrics_df = pd.DataFrame([
        {"Split": "prior_baseline_train_test_combined", **met_base},
        {"Split": "train", **met_tr},
        {"Split": "test", **met_te},
    ])

    metrics_df.to_csv(
        os.path.join(output_dir, "scatter_metrics_summary.csv"),
        index=False
    )

    return png_path, pdf_path, metrics_df


# =========================================================
# 7. Main program
# =========================================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    modeling_mode, zone_dir_map = infer_modeling_mode(MODEL_OUTPUT_DIR)

    print("=" * 80)
    print("Rebuild prior/train/test predictions from saved models")
    print("=" * 80)
    print(f"MODEL_OUTPUT_DIR : {MODEL_OUTPUT_DIR}")
    print(f"CSV_PATH         : {CSV_PATH}")
    print(f"OUTPUT_DIR       : {OUTPUT_DIR}")
    print(f"Detected mode    : {modeling_mode}")
    print()

    if modeling_mode == "global":
        train_out, test_out, meta = rebuild_predictions_global(MODEL_OUTPUT_DIR)
    elif modeling_mode == "zonewise":
        train_out, test_out, meta = rebuild_predictions_zonewise(
            MODEL_OUTPUT_DIR,
            zone_dir_map
        )
    else:
        raise ValueError(f"Unsupported modeling_mode: {modeling_mode}")

    final_target = meta["final_target"]

    train_csv = os.path.join(OUTPUT_DIR, "train_with_pred_rebuilt.csv")
    test_csv = os.path.join(OUTPUT_DIR, "test_with_pred_rebuilt.csv")

    train_out.to_csv(train_csv, index=False)
    test_out.to_csv(test_csv, index=False)

    png_path, pdf_path, metrics_df = plot_prior_train_test_scatter(
        train_df=train_out,
        test_df=test_out,
        output_dir=OUTPUT_DIR,
        modeling_mode=modeling_mode,
        final_target=final_target
    )

    print("=" * 80)
    print("Done.")
    print(f"Train rebuilt CSV : {train_csv}")
    print(f"Test rebuilt CSV  : {test_csv}")
    print(f"Figure PNG        : {png_path}")
    print(f"Figure PDF        : {pdf_path}")
    print()
    print(metrics_df.to_string(index=False))
    print("=" * 80)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
pCO2 inference script adapted to the latest target-adaptive training code.

Main additions in this version
------------------------------
1) Before prediction, strictly validate whether the input CSV contains all
   raw columns needed to reproduce the trained model's final feature list.
2) Validation respects feature engineering:
   - direct/raw features must exist in CSV
   - engineered features are validated against their dependency columns
   - internally generated deterministic features
     (geo_nx/geo_ny/geo_nz/month_sin/month_cos/months_since_1950)
     do NOT need to pre-exist in CSV
3) If any required raw dependency is missing, raise an error and stop.
4) Robust CSV reader retained.
5) Compatible with:
   - MODELING_MODE = "zonewise" / "global"
   - FINAL_TARGET = "pCO2" / "r_star_T"
   - ensemble members from latest training output

Important
---------
This script assumes the training metadata.pkl contains:
- features
- feature_engineering_enabled
- chla_offset
- feature_stats
- resolved_columns
which is consistent with the training code.
"""

import os
import re
import glob
import json
import warnings
warnings.filterwarnings("ignore")

# ========= threads (set before importing numeric libs) =========
DEFAULT_THREADS = 24
THREADS = int(os.getenv("PRED_THREADS", str(DEFAULT_THREADS)))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import numpy as np
import pandas as pd
from functools import lru_cache
import joblib
from tqdm.auto import tqdm
from catboost import CatBoostRegressor
from scipy.spatial import cKDTree
from netCDF4 import Dataset


# =========================================================
# 0. Config
# =========================================================
INPUT_CSV  = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN.csv"

# ---------------------------------------------------------
# Model location and mode
# ---------------------------------------------------------
TRAIN_OUTPUT_ROOT = "/data/wang/Result_pCO2/models_ML/current_model"

# {"auto", "zonewise", "global"}
MODELING_MODE = "zonewise"

if MODELING_MODE == "zonewise":
    OUTPUT_CSV = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_pred.csv"
elif MODELING_MODE == "global":
    OUTPUT_CSV = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_predglobal.csv"
else:
    # In auto mode, set a temporary value first and overwrite it later in main() according to resolved_mode.
    OUTPUT_CSV = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_pred.csv"

# Optional manual override.
MODELS_ROOT_OVERRIDE = None

# ---------------------------------------------------------
# External remap-mask control
# ---------------------------------------------------------
REMAP_MASK_NC = "/data/wang/mask_lineofsight.nc"

# ---------------------------------------------------------
# Columns
# ---------------------------------------------------------
ZONE_COL = "Zone"
TIME_COL = "Year"
MONTH_COL = "Month"
LAT_COL = "Latitude"
LON_COL = "Longitude"

PRED_COL = "pCO2_pred"
PRED_STD_COL = "pCO2_pred_std"

WRITE_TARGET_PRED = True
WRITE_ZONE_USED = True
WRITE_REMAP_DETAILS = True
WRITE_BOUNDARY_MASK_DETAILS = True

TARGET_PRED_COL = "target_pred"
TARGET_PRED_STD_COL = "target_pred_std"
ZONE_USED_COL = "Zone_used_for_model"
MODELING_MODE_USED_COL = "Modeling_mode_used"
MODEL_SCOPE_USED_COL = "Model_scope_used"

YEAR_MIN_OUTPUT = 1982

# ---------------------------------------------------------
# Remap-style boundary smoothing (zonewise only)
# ---------------------------------------------------------
ENABLE_REMAP_SMOOTHING = True
REMAP_MAX_MODELS = 3
REMAP_SIGMA_KM = 180.0
ALLOW_FALLBACK_TO_NEAREST_ZONE = True
MIN_ZONE_FOOTPRINT_POINTS = 5


if not ENABLE_REMAP_SMOOTHING:
    OUTPUT_CSV = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_predNoRemap.csv"
# ---------------------------------------------------------
# Numeric / output
# ---------------------------------------------------------
ROUND_DECIMALS = 3
EARTH_RADIUS_KM = 6371.0088

# If True, also check that rows used for prediction do not contain NaN in the
# required raw dependencies for the model features.
STRICT_ROWWISE_FINITE_CHECK = False


# =========================================================
# 1. General helpers
# =========================================================
def normalize_zone_value(v):
    if pd.isna(v):
        return np.nan
    try:
        fv = float(v)
        if not np.isfinite(fv):
            return np.nan
        if abs(fv - round(fv)) < 1e-9:
            return str(int(round(fv)))
        return f"{fv:g}"
    except Exception:
        s = str(v).strip()
        return s if s != "" else np.nan


def build_zone_key_series(series):
    return series.apply(normalize_zone_value)


def zone_dir_name(zone_key):
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", str(zone_key))
    return f"Zone_{safe}"


def find_first_existing(columns, candidates, required=False, field_desc="field"):
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise KeyError(f"Missing required {field_desc}. Candidates: {candidates}")
    return None


def compute_chla_offset(x):
    x = np.asarray(x, float)
    pos = x[np.isfinite(x) & (x > 0)]
    if pos.size == 0:
        return 1e-6
    p5 = np.nanpercentile(pos, 5)
    return max(1e-6, 0.01 * p5)


def get_log10_chla(df, chla_offset=None):
    if "log10_Chla" in df.columns:
        return pd.to_numeric(df["log10_Chla"], errors="coerce")
    if "Chla" not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)
    z = pd.to_numeric(df["Chla"], errors="coerce").to_numpy(dtype=float, copy=False)
    if chla_offset is None:
        chla_offset = compute_chla_offset(z)
    out = np.where(np.isfinite(z), np.log10(np.maximum(z, 0.0) + float(chla_offset)), np.nan)
    return pd.Series(out, index=df.index, dtype=float)


def prepare_time_features(df):
    out = df.copy()

    out[TIME_COL] = pd.to_numeric(out[TIME_COL], errors="coerce")
    out[MONTH_COL] = pd.to_numeric(out[MONTH_COL], errors="coerce")

    if "month_sin" not in out.columns:
        out["month_sin"] = np.sin(2.0 * np.pi * (out[MONTH_COL] - 1.0) / 12.0)
    if "month_cos" not in out.columns:
        out["month_cos"] = np.cos(2.0 * np.pi * (out[MONTH_COL] - 1.0) / 12.0)

    if "months_since_1950" not in out.columns:
        yy = pd.to_numeric(out[TIME_COL], errors="coerce").to_numpy(dtype=float)
        mm = pd.to_numeric(out[MONTH_COL], errors="coerce").to_numpy(dtype=float)
        out["months_since_1950"] = (yy - 1950.0) * 12.0 + (mm - 1.0)

    return out


def prepare_geo_features(df):
    out = df.copy()

    lat = pd.to_numeric(out[LAT_COL], errors="coerce").to_numpy(dtype=float, copy=False)
    lon = pd.to_numeric(out[LON_COL], errors="coerce").to_numpy(dtype=float, copy=False)

    lon_m180_180 = ((lon + 180.0) % 360.0) - 180.0
    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon_m180_180)

    if "geo_nx" not in out.columns:
        out["geo_nx"] = np.cos(lat_rad) * np.cos(lon_rad)
    if "geo_ny" not in out.columns:
        out["geo_ny"] = np.cos(lat_rad) * np.sin(lon_rad)
    if "geo_nz" not in out.columns:
        out["geo_nz"] = np.sin(lat_rad)

    return out


def transform_features(df_part, feature_cols, chla_offset, feature_stats):
    """
    Must mirror the current training code.
    """
    X = pd.DataFrame(index=df_part.index)

    sst = pd.to_numeric(df_part["SST"], errors="coerce") if "SST" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    sss = pd.to_numeric(df_part["SSS"], errors="coerce") if "SSS" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    do = pd.to_numeric(df_part["DO"], errors="coerce") if "DO" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    sst_anom = pd.to_numeric(df_part["SST_anom"], errors="coerce") if "SST_anom" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    do_anom = pd.to_numeric(df_part["DO_anom"], errors="coerce") if "DO_anom" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    nino34 = pd.to_numeric(df_part["NINO34"], errors="coerce") if "NINO34" in df_part.columns else pd.Series(np.nan, index=df_part.index)
    log10_chla = get_log10_chla(df_part, chla_offset)

    sst_c = sst - float(feature_stats.get("SST_mean", np.nan))
    sss_c = sss - float(feature_stats.get("SSS_mean", np.nan))
    do_c = do - float(feature_stats.get("DO_mean", np.nan))
    chla_c = log10_chla - float(feature_stats.get("log10_Chla_mean", np.nan))

    for c in feature_cols:
        if c == "log10_Chla":
            X[c] = log10_chla
        elif c == "SSTc_x_DOc":
            X[c] = sst_c * do_c
        elif c == "SSTc_x_SSSc":
            X[c] = sst_c * sss_c
        elif c == "SSTanom_x_DOanom":
            X[c] = sst_anom * do_anom
        elif c == "SSTanom_x_NINO34":
            X[c] = sst_anom * nino34
        elif c == "DOc_x_log10Chla":
            X[c] = do_c * chla_c
        else:
            if c in df_part.columns:
                X[c] = pd.to_numeric(df_part[c], errors="coerce")
            else:
                X[c] = np.nan
    return X


def unit_sphere_xyz(lat_deg, lon_deg):
    lat = np.deg2rad(np.asarray(lat_deg, dtype=float))
    lon = np.deg2rad(np.asarray(lon_deg, dtype=float))
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    return np.column_stack([x, y, z])


def chord_to_km(chord_dist):
    chord_dist = np.asarray(chord_dist, dtype=float)
    chord_dist = np.clip(chord_dist, 0.0, 2.0)
    angle = 2.0 * np.arcsin(np.clip(chord_dist / 2.0, 0.0, 1.0))
    return EARTH_RADIUS_KM * angle


def lon_to_0_360(lon_deg):
    x = np.asarray(lon_deg, np.float64)
    out = np.mod(x, 360.0)
    out[out < 0] += 360.0
    return out


def normalize_lon_to_m180_180(lon_arr):
    lon_arr = np.asarray(lon_arr, dtype=np.float64)
    return (lon_arr + 180.0) % 360.0 - 180.0


def read_csv_robust(path):
    attempts = [
        dict(engine="c", low_memory=False),
        dict(engine="python", low_memory=False),
        dict(engine="python", low_memory=False, on_bad_lines="skip"),
        dict(engine="python", low_memory=False, encoding="utf-8"),
        dict(engine="python", low_memory=False, encoding="utf-8", on_bad_lines="skip"),
        dict(engine="python", low_memory=False, encoding="latin1"),
        dict(engine="python", low_memory=False, encoding="latin1", on_bad_lines="skip"),
    ]

    last_err = None
    for i, kw in enumerate(attempts, start=1):
        try:
            print(f"[CSV] Attempt {i}: pandas.read_csv({kw})")
            df = pd.read_csv(path, **kw)
            print(f"[CSV] Success with attempt {i}. Shape = {df.shape}")
            return df
        except Exception as e:
            last_err = e
            print(f"[CSV] Attempt {i} failed: {repr(e)}")

    raise RuntimeError(f"Failed to read CSV robustly: {path}\nLast error: {repr(last_err)}")


# =========================================================
# 1a. Feature dependency validation
# =========================================================
INTERNAL_GENERATED_FEATURES = {
    "geo_nx", "geo_ny", "geo_nz",
    "month_sin", "month_cos",
    "months_since_1950",
}

# These are not necessarily present in raw CSV, but the script can create them
# from existing raw columns.
RAW_COLUMNS_NEEDED_TO_GENERATE_INTERNAL = {
    "geo_nx": {LAT_COL, LON_COL},
    "geo_ny": {LAT_COL, LON_COL},
    "geo_nz": {LAT_COL},
    "month_sin": {MONTH_COL},
    "month_cos": {MONTH_COL},
    "months_since_1950": {TIME_COL, MONTH_COL},
}

ENGINEERED_FEATURE_DEPENDENCIES = {
    "log10_Chla": {"Chla"},
    "SSTc_x_DOc": {"SST", "DO"},
    "SSTc_x_SSSc": {"SST", "SSS"},
    "SSTanom_x_DOanom": {"SST_anom", "DO_anom"},
    "SSTanom_x_NINO34": {"SST_anom", "NINO34"},
    "DOc_x_log10Chla": {"DO", "Chla"},
}


def get_required_raw_columns_for_feature(feature_name):
    """
    Return the raw/base columns required to reproduce one model feature.
    """
    # Deterministic internal features created by the script
    if feature_name in INTERNAL_GENERATED_FEATURES:
        return set(RAW_COLUMNS_NEEDED_TO_GENERATE_INTERNAL.get(feature_name, set()))

    # Engineered features
    if feature_name in ENGINEERED_FEATURE_DEPENDENCIES:
        return set(ENGINEERED_FEATURE_DEPENDENCIES[feature_name])

    # Otherwise treat it as a direct/raw feature that must be present
    return {feature_name}


def collect_required_raw_columns_from_feature_list(feature_list):
    req = set()
    per_feature_req = {}
    for f in feature_list:
        deps = get_required_raw_columns_for_feature(f)
        per_feature_req[f] = set(deps)
        req.update(deps)
    return req, per_feature_req


def validate_required_inputs_before_prediction(df, model_obj, scope_desc="model"):
    """
    Hard validation before prediction:
    - final trained feature list must be recoverable from input CSV
    - engineered features are checked against their base dependencies
    """
    features = list(model_obj.get("features", []) or [])
    if len(features) == 0:
        raise RuntimeError(f"[{scope_desc}] metadata has empty 'features' list; cannot validate/predict.")

    req_raw_cols, per_feature_req = collect_required_raw_columns_from_feature_list(features)

    missing_cols = sorted([c for c in req_raw_cols if c not in df.columns])
    if missing_cols:
        lines = [f"[{scope_desc}] Input CSV is missing required raw columns for trained model features:"]
        for feat in features:
            deps = sorted(per_feature_req[feat])
            miss = [x for x in deps if x not in df.columns]
            if len(miss) > 0:
                lines.append(f"  - feature '{feat}' requires {deps}, but missing {miss}")
        raise KeyError("\n".join(lines))

    # Additional target-reconstruction validation
    final_target = str(model_obj.get("final_target", "")).strip()
    resolved = model_obj.get("resolved_columns", {}) or {}

    if final_target == "r_star_T":
        prior_clim_candidates = []
        delta_t_candidates = []

        rc_prior = resolved.get("prior_clim_col", None)
        rc_delta = resolved.get("delta_pCO2_T_col", None)

        if rc_prior is not None:
            prior_clim_candidates.append(rc_prior)
        if rc_delta is not None:
            delta_t_candidates.append(rc_delta)

        prior_clim_candidates += [
            "pCO2_prior_clim_mapped", "pco2_prior_clim_mapped",
            "pCO2_prior_clim", "pco2_prior_clim"
        ]
        delta_t_candidates += ["delta_pCO2_T"]

        _ = find_first_existing(df.columns, prior_clim_candidates, required=True, field_desc="pCO2 prior climatology for r_star_T reconstruction")
        _ = find_first_existing(df.columns, delta_t_candidates, required=True, field_desc="delta_pCO2_T for r_star_T reconstruction")

    # Optional row-wise finite check
    if STRICT_ROWWISE_FINITE_CHECK:
        check_cols = sorted(req_raw_cols.intersection(set(df.columns)))
        bad_mask = pd.Series(False, index=df.index)
        for c in check_cols:
            bad_mask |= pd.to_numeric(df[c], errors="coerce").isna()
        n_bad = int(bad_mask.sum())
        if n_bad > 0:
            raise ValueError(
                f"[{scope_desc}] Found {n_bad} rows with NaN/non-numeric values in required raw columns: {check_cols}. "
                "Set STRICT_ROWWISE_FINITE_CHECK=False if you prefer allowing CatBoost input with NaNs."
            )

    print(f"[Validate] {scope_desc}: feature validation passed.")
    print(f"[Validate] {scope_desc}: trained features = {features}")
    print(f"[Validate] {scope_desc}: required raw/base columns = {sorted(req_raw_cols)}")


def validate_all_scope_models_against_input(df, resolved_mode, models_root):
    """
    Validate every model that may be used.
    For global mode: validate the global model.
    For zonewise mode: validate all available zone models, because remap blending
    may call neighboring zone models too.
    """
    if resolved_mode == "global":
        model_obj = load_global_model(models_root)
        validate_required_inputs_before_prediction(df, model_obj, scope_desc="GLOBAL")
        return

    zone_models = scan_available_zone_models(models_root)
    if len(zone_models) == 0:
        raise FileNotFoundError(f"No usable Zone_* models found under: {models_root}")

    for zk, model_obj in zone_models.items():
        validate_required_inputs_before_prediction(df, model_obj, scope_desc=f"Zone_{zk}")


# =========================================================
# 1b. External remap-mask helpers
# =========================================================
_REMAP_MASK_CACHE = {
    "path": None,
    "lat": None,
    "lon": None,
    "remap_mask_los": None,
    "dist2adjboundary_km": None,
    "lat_sorted": None,
    "lon_sorted": None,
    "lat_order": None,
    "lon_order": None,
}


def _as_sorted_axis_and_index(axis):
    axis = np.asarray(axis, np.float64)
    order = np.argsort(axis)
    sorted_axis = axis[order]
    return sorted_axis, order


def nearest_index_on_sorted_axis(sorted_axis, values):
    v = np.asarray(values, np.float64)
    idx = np.searchsorted(sorted_axis, v, side="left")
    idx0 = np.clip(idx - 1, 0, sorted_axis.size - 1)
    idx1 = np.clip(idx, 0, sorted_axis.size - 1)
    choose_right = np.abs(sorted_axis[idx1] - v) < np.abs(sorted_axis[idx0] - v)
    return np.where(choose_right, idx1, idx0).astype(np.int64)


def load_remap_mask_nc(mask_nc_path):
    global _REMAP_MASK_CACHE
    if _REMAP_MASK_CACHE["path"] == mask_nc_path:
        return

    with Dataset(mask_nc_path, "r") as nc:
        lat = nc["lat"][:].astype(np.float64)
        lon = nc["lon"][:].astype(np.float64)
        remap_mask_los = nc["remap_mask_los"][:].astype(np.uint8)
        dist2adjboundary_km = nc["dist2adjboundary_km"][:].astype(np.float32)

    lat_sorted, lat_order = _as_sorted_axis_and_index(lat)
    lon_sorted, lon_order = _as_sorted_axis_and_index(lon)

    _REMAP_MASK_CACHE = {
        "path": mask_nc_path,
        "lat": lat,
        "lon": lon,
        "remap_mask_los": remap_mask_los,
        "dist2adjboundary_km": dist2adjboundary_km,
        "lat_sorted": lat_sorted,
        "lon_sorted": lon_sorted,
        "lat_order": lat_order,
        "lon_order": lon_order,
    }


def map_remap_mask_to_points(mask_nc_path, lat_points, lon_points):
    load_remap_mask_nc(mask_nc_path)

    lat_sorted = _REMAP_MASK_CACHE["lat_sorted"]
    lon_sorted = _REMAP_MASK_CACHE["lon_sorted"]
    lat_order = _REMAP_MASK_CACHE["lat_order"]
    lon_order = _REMAP_MASK_CACHE["lon_order"]

    remap_mask_los = _REMAP_MASK_CACHE["remap_mask_los"]
    dist2adjboundary_km = _REMAP_MASK_CACHE["dist2adjboundary_km"]
    lon_nc = _REMAP_MASK_CACHE["lon"]

    lat_points = np.asarray(lat_points, np.float64)
    lon_points = np.asarray(lon_points, np.float64)

    if np.nanmax(lon_nc) > 180.0:
        lon_query = lon_to_0_360(lon_points)
    else:
        lon_query = normalize_lon_to_m180_180(lon_points)

    idx_lat_sorted = nearest_index_on_sorted_axis(lat_sorted, lat_points)
    idx_lon_sorted = nearest_index_on_sorted_axis(lon_sorted, lon_query)

    lat_idx = lat_order[idx_lat_sorted]
    lon_idx = lon_order[idx_lon_sorted]

    blend_allow = remap_mask_los[lat_idx, lon_idx].astype(np.uint8)
    boundary_dist = dist2adjboundary_km[lat_idx, lon_idx].astype(np.float32)

    return blend_allow, boundary_dist


# =========================================================
# 2. Model-root resolution
# =========================================================
def resolve_model_root_and_mode():
    if MODELS_ROOT_OVERRIDE is not None:
        root = MODELS_ROOT_OVERRIDE
        if not os.path.isdir(root):
            raise FileNotFoundError(f"MODELS_ROOT_OVERRIDE does not exist: {root}")
        mode = infer_modeling_mode_from_root(root)
        return root, mode

    mode = MODELING_MODE.lower().strip()
    if mode not in {"auto", "zonewise", "global"}:
        raise ValueError(f"Invalid MODELING_MODE={MODELING_MODE}. Must be one of ['auto','zonewise','global'].")

    zone_root = os.path.join(TRAIN_OUTPUT_ROOT, "zonewise_model")
    global_root = os.path.join(TRAIN_OUTPUT_ROOT, "global_model")

    if mode == "zonewise":
        if not os.path.isdir(zone_root):
            raise FileNotFoundError(f"Zonewise model root not found: {zone_root}")
        return zone_root, "zonewise"

    if mode == "global":
        if not os.path.isdir(global_root):
            raise FileNotFoundError(f"Global model root not found: {global_root}")
        return global_root, "global"

    zone_ok = os.path.isdir(zone_root) and len(glob.glob(os.path.join(zone_root, "Zone_*"))) > 0
    global_ok = os.path.isdir(global_root) and os.path.exists(os.path.join(global_root, "metadata.pkl"))

    if zone_ok and not global_ok:
        return zone_root, "zonewise"
    if global_ok and not zone_ok:
        return global_root, "global"
    if zone_ok and global_ok:
        zone_mtime = os.path.getmtime(zone_root)
        global_mtime = os.path.getmtime(global_root)
        if zone_mtime >= global_mtime:
            return zone_root, "zonewise"
        return global_root, "global"

    raise FileNotFoundError(
        "Could not auto-detect trained model root under TRAIN_OUTPUT_ROOT. "
        f"Checked:\n  {zone_root}\n  {global_root}"
    )


def infer_modeling_mode_from_root(root):
    if os.path.exists(os.path.join(root, "metadata.pkl")) and os.path.isdir(os.path.join(root, "ensemble_members")):
        try:
            meta = joblib.load(os.path.join(root, "metadata.pkl"))
            mm = str(meta.get("modeling_mode", "")).strip().lower()
            if mm in {"zonewise", "global"}:
                return mm
        except Exception:
            pass

    if len(glob.glob(os.path.join(root, "Zone_*"))) > 0:
        return "zonewise"

    return "global"


# =========================================================
# 3. Model loading
# =========================================================
def list_member_model_paths(scope_dir):
    member_dir = os.path.join(scope_dir, "ensemble_members")
    member_paths = sorted(glob.glob(os.path.join(member_dir, "member_*.cbm")))
    if len(member_paths) > 0:
        return member_paths

    single_model = os.path.join(scope_dir, "model.cbm")
    if os.path.exists(single_model):
        return [single_model]

    return []


@lru_cache(maxsize=256)
def load_scope_model(scope_dir):
    meta_path = os.path.join(scope_dir, "metadata.pkl")
    if not os.path.exists(meta_path):
        return None

    meta = joblib.load(meta_path)
    feat_order = list(meta.get("features", []) or [])
    chla_offset = meta.get("chla_offset", None)
    feature_stats = meta.get("feature_stats", {}) or {}

    member_paths = list_member_model_paths(scope_dir)
    if len(member_paths) == 0:
        return None

    models = []
    for mp in member_paths:
        mdl = CatBoostRegressor()
        mdl.load_model(mp)
        models.append(mdl)

    return {
        "scope_dir": scope_dir,
        "meta": meta,
        "models": models,
        "member_paths": member_paths,
        "features": feat_order,
        "chla_offset": chla_offset,
        "feature_stats": feature_stats,
        "final_target": str(meta.get("final_target", "")).strip(),
        "modeling_mode": str(meta.get("modeling_mode", "")).strip().lower(),
        "resolved_columns": meta.get("resolved_columns", {}) or {},
        "feature_engineering_enabled": bool(meta.get("feature_engineering_enabled", True)),
    }


def load_global_model(models_root):
    obj = load_scope_model(models_root)
    if obj is None:
        raise FileNotFoundError(f"Global metadata/model not found under: {models_root}")
    return obj


def scan_available_zone_models(models_root):
    zone_dirs = sorted(glob.glob(os.path.join(models_root, "Zone_*")))
    zone_models = {}
    for zdir in zone_dirs:
        obj = load_scope_model(zdir)
        if obj is None:
            continue
        zone_key = obj["meta"].get("zone_key", None)
        if zone_key is None:
            zone_key = os.path.basename(zdir).replace("Zone_", "", 1)
        zone_key = normalize_zone_value(zone_key)
        if pd.isna(zone_key):
            continue
        zone_models[str(zone_key)] = obj
    return zone_models


# =========================================================
# 4. Prediction core
# =========================================================
def build_feature_matrix(df_part, model_obj):
    feat_order = model_obj["features"]
    if not feat_order:
        raise RuntimeError("Empty feature list in metadata; cannot predict.")
    chla_offset = model_obj.get("chla_offset", None)
    feature_stats = model_obj.get("feature_stats", {}) or {}
    X_df = transform_features(df_part, feat_order, chla_offset, feature_stats)
    X_df = X_df.reindex(columns=feat_order, fill_value=np.nan)
    return X_df


def predict_ensemble_mean_std(df_part, model_obj):
    X_df = build_feature_matrix(df_part, model_obj)
    X = X_df.to_numpy(dtype=float, copy=False)

    member_preds = []
    for mdl in model_obj["models"]:
        yhat = mdl.predict(X, thread_count=THREADS)
        member_preds.append(np.asarray(yhat, dtype=float))

    arr = np.column_stack(member_preds)
    mu = np.nanmean(arr, axis=1)
    sd = np.nanstd(arr, axis=1, ddof=0)
    return mu, sd


def find_offset_for_r_star_T(df_part, model_obj):
    resolved = model_obj.get("resolved_columns", {}) or {}

    prior_clim_candidates = []
    delta_t_candidates = []

    rc_prior = resolved.get("prior_clim_col", None)
    rc_delta = resolved.get("delta_pCO2_T_col", None)

    if rc_prior is not None:
        prior_clim_candidates.append(rc_prior)
    if rc_delta is not None:
        delta_t_candidates.append(rc_delta)

    prior_clim_candidates += [
        "pCO2_prior_clim_mapped", "pco2_prior_clim_mapped",
        "pCO2_prior_clim", "pco2_prior_clim"
    ]
    delta_t_candidates += ["delta_pCO2_T"]

    prior_clim_col = find_first_existing(df_part.columns, prior_clim_candidates, required=True, field_desc="pCO2 prior climatology")
    delta_t_col = find_first_existing(df_part.columns, delta_t_candidates, required=True, field_desc="delta_pCO2_T")

    prior_clim = pd.to_numeric(df_part[prior_clim_col], errors="coerce").to_numpy(dtype=float, copy=False)
    delta_t = pd.to_numeric(df_part[delta_t_col], errors="coerce").to_numpy(dtype=float, copy=False)
    return prior_clim + delta_t


def reconstruct_pco2_from_target(df_part, target_mu, target_sd, model_obj):
    final_target = str(model_obj.get("final_target", "")).strip()

    if final_target == "pCO2":
        return target_mu.copy(), target_sd.copy()

    if final_target == "r_star_T":
        offset = find_offset_for_r_star_T(df_part, model_obj)
        return offset + target_mu, target_sd.copy()

    raise ValueError(f"Unsupported final_target in metadata: {final_target}")


# =========================================================
# 5. Remap-style boundary smoothing
# =========================================================
def build_zone_footprint_trees(df, available_zone_keys):
    zone_trees = {}
    zone_centers = {}

    tmp = df[[LAT_COL, LON_COL, "_ZoneKey"]].copy()
    tmp[LAT_COL] = pd.to_numeric(tmp[LAT_COL], errors="coerce")
    tmp[LON_COL] = pd.to_numeric(tmp[LON_COL], errors="coerce")
    tmp = tmp.dropna(subset=[LAT_COL, LON_COL, "_ZoneKey"])

    for zk in available_zone_keys:
        sub = tmp[tmp["_ZoneKey"] == zk][[LAT_COL, LON_COL]].drop_duplicates()
        if len(sub) < int(MIN_ZONE_FOOTPRINT_POINTS):
            continue

        lat = sub[LAT_COL].to_numpy(dtype=float, copy=False)
        lon = sub[LON_COL].to_numpy(dtype=float, copy=False)
        xyz = unit_sphere_xyz(lat, lon)
        tree = cKDTree(xyz)

        zone_trees[zk] = {
            "tree": tree,
            "n_points": int(len(sub)),
        }
        zone_centers[zk] = (
            float(np.nanmean(lat)),
            float(np.nanmean(lon)),
        )

    return zone_trees, zone_centers


def build_unique_spatial_table(df):
    tmp = df[[LAT_COL, LON_COL, "_ZoneKey"]].copy()
    tmp[LAT_COL] = pd.to_numeric(tmp[LAT_COL], errors="coerce")
    tmp[LON_COL] = pd.to_numeric(tmp[LON_COL], errors="coerce")
    tmp = tmp.dropna(subset=[LAT_COL, LON_COL])

    spatial = tmp.groupby([LAT_COL, LON_COL], as_index=False).first()
    spatial["_SpatialID"] = np.arange(len(spatial), dtype=np.int64)
    return spatial


def compute_remap_candidates_for_unique_points(spatial_df, zone_trees, primary_zone_series):
    available_zones = list(zone_trees.keys())
    if len(available_zones) == 0:
        raise RuntimeError("No valid zone footprint trees were built.")

    xyz_q = unit_sphere_xyz(
        spatial_df[LAT_COL].to_numpy(dtype=float, copy=False),
        spatial_df[LON_COL].to_numpy(dtype=float, copy=False)
    )

    dist_km_mat = np.full((len(spatial_df), len(available_zones)), np.inf, dtype=float)

    for j, zk in enumerate(available_zones):
        d_chord, _ = zone_trees[zk]["tree"].query(xyz_q, k=1, workers=-1)
        dist_km_mat[:, j] = chord_to_km(d_chord)

    cand_zones_list = []
    cand_weights_list = []
    primary_available_list = []

    primary_keys = primary_zone_series.astype("object").tolist()
    blend_allow_flags = spatial_df["_BlendAllowedByMask"].astype(np.uint8).tolist()

    for i in range(len(spatial_df)):
        drow = dist_km_mat[i, :]
        primary_zk = primary_keys[i]
        primary_available = (primary_zk in zone_trees)
        primary_available_list.append(bool(primary_available))

        if not primary_available:
            if not ALLOW_FALLBACK_TO_NEAREST_ZONE:
                cand_zones_list.append([])
                cand_weights_list.append([])
                continue

            j_nearest = int(np.argmin(drow))
            chosen_zones = [available_zones[j_nearest]]
            raw_w = np.array([1.0], dtype=float)
            w = raw_w / raw_w.sum()

            cand_zones_list.append(chosen_zones)
            cand_weights_list.append(w.tolist())
            continue

        j_primary = available_zones.index(primary_zk)

        if int(blend_allow_flags[i]) == 0:
            chosen_zones = [primary_zk]
            raw_w = np.array([1.0], dtype=float)
            w = raw_w / raw_w.sum()

            cand_zones_list.append(chosen_zones)
            cand_weights_list.append(w.tolist())
            continue

        other_idx = [j for j in range(len(available_zones)) if j != j_primary]
        other_idx = sorted(other_idx, key=lambda j: drow[j])

        max_neighbors = max(0, int(REMAP_MAX_MODELS) - 1)
        neighbor_idx = other_idx[:max_neighbors]

        chosen_idx = [j_primary] + neighbor_idx
        chosen_idx = sorted(chosen_idx, key=lambda j: drow[j])

        chosen_zones = [available_zones[j] for j in chosen_idx]
        chosen_dist = np.array([drow[j] for j in chosen_idx], dtype=float)

        if not ENABLE_REMAP_SMOOTHING:
            chosen_zones = [primary_zk]
            chosen_dist = np.array([drow[j_primary]], dtype=float)

        sigma = max(1.0, float(REMAP_SIGMA_KM))
        raw_w = np.exp(-0.5 * (chosen_dist / sigma) ** 2)

        if not np.isfinite(raw_w).any() or np.nansum(raw_w) <= 0:
            raw_w = np.zeros_like(chosen_dist)
            raw_w[np.argmin(chosen_dist)] = 1.0

        w = raw_w / np.nansum(raw_w)

        cand_zones_list.append(chosen_zones)
        cand_weights_list.append(w.tolist())

    out = spatial_df.copy()
    out["_CandidateZones"] = cand_zones_list
    out["_CandidateWeights"] = cand_weights_list
    out["_PrimaryZoneModelAvailable"] = primary_available_list
    return out


def weighted_mixture_mean_std(means, stds, weights):
    means = np.asarray(means, dtype=float)
    stds = np.asarray(stds, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mu = np.sum(weights * means, axis=0)
    second = np.sum(weights * (stds**2 + means**2), axis=0)
    var = np.maximum(0.0, second - mu**2)
    sd = np.sqrt(var)
    return mu, sd


# =========================================================
# 6. Main pipelines
# =========================================================
def prepare_input_dataframe(df, need_zone):
    required = [TIME_COL, MONTH_COL, LAT_COL, LON_COL]
    if need_zone:
        required.append(ZONE_COL)

    miss = [c for c in required if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns: {miss}")

    out = df.copy()
    out = prepare_time_features(out)
    out = prepare_geo_features(out)
    out[TIME_COL] = pd.to_numeric(out[TIME_COL], errors="coerce")
    out[MONTH_COL] = pd.to_numeric(out[MONTH_COL], errors="coerce")
    out[LAT_COL] = pd.to_numeric(out[LAT_COL], errors="coerce")
    out[LON_COL] = pd.to_numeric(out[LON_COL], errors="coerce")

    if need_zone:
        out["_ZoneKey"] = build_zone_key_series(out[ZONE_COL])
    else:
        out["_ZoneKey"] = "GLOBAL"

    return out


def run_global_inference(df, models_root):
    model_obj = load_global_model(models_root)

    target_mu, target_sd = predict_ensemble_mean_std(df, model_obj)
    pco2_mu, pco2_sd = reconstruct_pco2_from_target(df, target_mu, target_sd, model_obj)

    out = df.copy()
    out[PRED_COL] = np.round(pco2_mu.astype(np.float64), ROUND_DECIMALS)
    out[PRED_STD_COL] = np.round(pco2_sd.astype(np.float64), ROUND_DECIMALS)

    if WRITE_TARGET_PRED:
        out[TARGET_PRED_COL] = np.round(target_mu.astype(np.float64), ROUND_DECIMALS)
        out[TARGET_PRED_STD_COL] = np.round(target_sd.astype(np.float64), ROUND_DECIMALS)

    if WRITE_ZONE_USED:
        out[ZONE_USED_COL] = "GLOBAL"

    out[MODELING_MODE_USED_COL] = "global"
    out[MODEL_SCOPE_USED_COL] = "GLOBAL"
    out["Final_target_used"] = str(model_obj.get("final_target", ""))
    out["Ensemble_members_used"] = int(len(model_obj["models"]))

    return out


def run_zonewise_inference(df, models_root):
    zone_models = scan_available_zone_models(models_root)
    if len(zone_models) == 0:
        raise FileNotFoundError(f"No usable Zone_* models found under: {models_root}")

    available_zone_keys = sorted(
        zone_models.keys(),
        key=lambda x: (0, float(x)) if re.fullmatch(r"-?\d+(\.\d+)?", str(x)) else (1, str(x))
    )

    print(f"[Zonewise] available trained zone models: {available_zone_keys}")

    zone_trees, zone_centers = build_zone_footprint_trees(df, available_zone_keys)
    if len(zone_trees) == 0:
        raise RuntimeError("No valid zone footprint trees could be built from the prediction dataframe.")

    spatial_df = build_unique_spatial_table(df)

    blend_allow, boundary_dist = map_remap_mask_to_points(
        REMAP_MASK_NC,
        spatial_df[LAT_COL].to_numpy(dtype=float, copy=False),
        spatial_df[LON_COL].to_numpy(dtype=float, copy=False)
    )

    spatial_df["_BlendAllowedByMask"] = blend_allow
    spatial_df["_BoundaryDistKM"] = boundary_dist

    spatial_df = compute_remap_candidates_for_unique_points(
        spatial_df=spatial_df,
        zone_trees=zone_trees,
        primary_zone_series=spatial_df["_ZoneKey"]
    )

    out = df.copy()
    out = out.merge(
        spatial_df[
            [
                LAT_COL, LON_COL,
                "_CandidateZones", "_CandidateWeights",
                "_PrimaryZoneModelAvailable",
                "_BlendAllowedByMask", "_BoundaryDistKM"
            ]
        ],
        on=[LAT_COL, LON_COL],
        how="left"
    )

    n = len(out)
    target_mu_final = np.full(n, np.nan, dtype=float)
    target_sd_final = np.full(n, np.nan, dtype=float)
    pco2_mu_final = np.full(n, np.nan, dtype=float)
    pco2_sd_final = np.full(n, np.nan, dtype=float)
    primary_zone_used = np.full(n, None, dtype=object)
    blended_zone_desc = np.full(n, None, dtype=object)
    blended_weight_desc = np.full(n, None, dtype=object)
    remap_applied = np.zeros(n, dtype=bool)

    zone_to_rowidx = {}
    for i, (cand_zones, cand_weights) in enumerate(zip(out["_CandidateZones"], out["_CandidateWeights"])):
        if not isinstance(cand_zones, list) or len(cand_zones) == 0:
            continue
        for zk in cand_zones:
            zone_to_rowidx.setdefault(zk, []).append(i)

    zone_pred_cache = {}
    for zk, ridx in tqdm(zone_to_rowidx.items(), desc="Predict needed zone models", unit="zone"):
        if zk not in zone_models:
            continue
        idx = np.array(sorted(set(ridx)), dtype=np.int64)
        sub = out.iloc[idx]
        model_obj = zone_models[zk]

        target_mu, target_sd = predict_ensemble_mean_std(sub, model_obj)
        pco2_mu, pco2_sd = reconstruct_pco2_from_target(sub, target_mu, target_sd, model_obj)

        zone_pred_cache[zk] = {
            "idx": idx,
            "target_mu": target_mu,
            "target_sd": target_sd,
            "pco2_mu": pco2_mu,
            "pco2_sd": pco2_sd,
            "final_target": model_obj.get("final_target", ""),
            "n_members": len(model_obj["models"]),
        }

    zone_local_lookup = {}
    for zk, obj in zone_pred_cache.items():
        zone_local_lookup[zk] = {int(ii): pos for pos, ii in enumerate(obj["idx"])}

    for i in tqdm(range(n), desc="Blend remap predictions", unit="row"):
        primary_zk = out.iloc[i]["_ZoneKey"]
        cand_zones = out.iloc[i]["_CandidateZones"]
        cand_weights = out.iloc[i]["_CandidateWeights"]

        if not isinstance(cand_zones, list) or len(cand_zones) == 0:
            continue

        mus_t, sds_t, mus_p, sds_p = [], [], [], []
        used_zones, used_weights = [], []

        for zk, w in zip(cand_zones, cand_weights):
            if zk not in zone_pred_cache:
                continue
            pos_map = zone_local_lookup[zk]
            if i not in pos_map:
                continue
            pos = pos_map[i]

            mus_t.append(zone_pred_cache[zk]["target_mu"][pos])
            sds_t.append(zone_pred_cache[zk]["target_sd"][pos])
            mus_p.append(zone_pred_cache[zk]["pco2_mu"][pos])
            sds_p.append(zone_pred_cache[zk]["pco2_sd"][pos])
            used_zones.append(zk)
            used_weights.append(float(w))

        if len(used_zones) == 0:
            continue

        used_weights = np.asarray(used_weights, dtype=float)
        used_weights = used_weights / np.sum(used_weights)

        mu_t, sd_t = weighted_mixture_mean_std(
            means=np.asarray(mus_t, dtype=float)[:, None],
            stds=np.asarray(sds_t, dtype=float)[:, None],
            weights=used_weights[:, None]
        )
        mu_p, sd_p = weighted_mixture_mean_std(
            means=np.asarray(mus_p, dtype=float)[:, None],
            stds=np.asarray(sds_p, dtype=float)[:, None],
            weights=used_weights[:, None]
        )

        target_mu_final[i] = float(mu_t[0])
        target_sd_final[i] = float(sd_t[0])
        pco2_mu_final[i] = float(mu_p[0])
        pco2_sd_final[i] = float(sd_p[0])

        primary_zone_used[i] = str(primary_zk) if not pd.isna(primary_zk) else None
        blended_zone_desc[i] = ",".join(map(str, used_zones))
        blended_weight_desc[i] = ",".join([f"{x:.6f}" for x in used_weights])
        remap_applied[i] = (len(used_zones) > 1)

    out[PRED_COL] = np.round(pco2_mu_final.astype(np.float64), ROUND_DECIMALS)
    out[PRED_STD_COL] = np.round(pco2_sd_final.astype(np.float64), ROUND_DECIMALS)

    if WRITE_TARGET_PRED:
        out[TARGET_PRED_COL] = np.round(target_mu_final.astype(np.float64), ROUND_DECIMALS)
        out[TARGET_PRED_STD_COL] = np.round(target_sd_final.astype(np.float64), ROUND_DECIMALS)

    if WRITE_ZONE_USED:
        out[ZONE_USED_COL] = primary_zone_used

    out[MODELING_MODE_USED_COL] = "zonewise"
    out[MODEL_SCOPE_USED_COL] = np.where(
        pd.Series(primary_zone_used).notna(),
        pd.Series(primary_zone_used).map(lambda z: f"Zone_{z}"),
        None
    )

    if WRITE_REMAP_DETAILS:
        out["Remap_applied"] = remap_applied
        out["Blended_zone_models"] = blended_zone_desc
        out["Blended_zone_weights"] = blended_weight_desc

    if WRITE_BOUNDARY_MASK_DETAILS:
        out["Remap_allowed_by_mask"] = out["_BlendAllowedByMask"]
        out["Boundary_dist_to_adjzone_km"] = np.round(
            pd.to_numeric(out["_BoundaryDistKM"], errors="coerce").to_numpy(dtype=float),
            ROUND_DECIMALS
        )

    out["Final_target_used"] = ""
    out["Ensemble_members_used"] = np.nan
    if len(zone_models) > 0:
        sample_model = next(iter(zone_models.values()))
        out["Final_target_used"] = str(sample_model.get("final_target", ""))
        out["Ensemble_members_used"] = int(len(sample_model["models"]))

    return out


# =========================================================
# 7. Main
# =========================================================
def main():
    if not os.path.exists(INPUT_CSV):
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

    models_root, resolved_mode = resolve_model_root_and_mode()

    if resolved_mode == "zonewise":
        if not os.path.exists(REMAP_MASK_NC):
            raise FileNotFoundError(f"REMAP_MASK_NC not found: {REMAP_MASK_NC}")

    print("=" * 80)
    print("Inference configuration")
    print("=" * 80)
    print(f"INPUT_CSV              : {INPUT_CSV}")
    print(f"OUTPUT_CSV             : {OUTPUT_CSV}")
    print(f"TRAIN_OUTPUT_ROOT      : {TRAIN_OUTPUT_ROOT}")
    print(f"MODELS_ROOT            : {models_root}")
    print(f"MODELING_MODE(request) : {MODELING_MODE}")
    print(f"MODELING_MODE(resolved): {resolved_mode}")
    print(f"REMAP_MASK_NC          : {REMAP_MASK_NC if resolved_mode == 'zonewise' else 'NOT_USED_IN_GLOBAL'}")
    print(f"ENABLE_REMAP_SMOOTHING : {ENABLE_REMAP_SMOOTHING}")
    print(f"REMAP_MAX_MODELS       : {REMAP_MAX_MODELS}")
    print(f"REMAP_SIGMA_KM         : {REMAP_SIGMA_KM}")
    print(f"YEAR_MIN_OUTPUT        : {YEAR_MIN_OUTPUT}")
    print("=" * 80)

    df_raw = read_csv_robust(INPUT_CSV)

    # Hard validation must happen before prediction.
    # We validate against the raw input CSV schema first.
    validate_all_scope_models_against_input(df_raw, resolved_mode, models_root)

    need_zone = (resolved_mode == "zonewise")
    df = prepare_input_dataframe(df_raw, need_zone=need_zone)
    df = df[pd.to_numeric(df[TIME_COL], errors="coerce") >= YEAR_MIN_OUTPUT].copy()

    if resolved_mode == "global":
        out = run_global_inference(df, models_root)
    else:
        out = run_zonewise_inference(df, models_root)

    helper_cols = [
        "_ZoneKey",
        "_CandidateZones",
        "_CandidateWeights",
        "_PrimaryZoneModelAvailable",
        "_BlendAllowedByMask",
        "_BoundaryDistKM",
    ]
    for c in helper_cols:
        if c in out.columns:
            out.drop(columns=c, inplace=True)

    out.to_csv(OUTPUT_CSV, index=False)

    n_valid = int(np.isfinite(pd.to_numeric(out[PRED_COL], errors="coerce").to_numpy(dtype=float)).sum())
    print("Done.")
    print("Input :", INPUT_CSV)
    print("Output:", OUTPUT_CSV)
    print(f"Valid predictions: {n_valid} / {len(out)}")

    if resolved_mode == "zonewise" and WRITE_REMAP_DETAILS and "Remap_applied" in out.columns:
        n_remap = int(pd.Series(out["Remap_applied"]).fillna(False).sum())
        print(f"Rows with multi-zone remap blending: {n_remap} / {len(out)}")

    if resolved_mode == "zonewise" and WRITE_BOUNDARY_MASK_DETAILS and "Remap_allowed_by_mask" in out.columns:
        n_allow = int(pd.Series(out["Remap_allowed_by_mask"]).fillna(0).astype(int).sum())
        print(f"Rows allowed to blend by external mask: {n_allow} / {len(out)}")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
make_monthly_pco2_netcdf_1deg_offshoremask_smooth_floor_singlefile_current.py

根据当前预测脚本输出的单个预测 CSV（散点）生成月平均表层 pCO2 NetCDF（单文件），
输出 1°×1° 网格，所有年月数据写入同一个 nc 文件的 time 维。

当前默认输入（与最新预测脚本一致）:
  /data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_pred.csv

当前优先支持的预测列：
  1) pCO2_pred                      （首选，当前预测脚本默认输出）
  2) target_pred + prior_clim + delta_pCO2_T
     （适配 FINAL_TARGET = "r_star_T" 的当前流程）
  3) target_pred                   （适配 FINAL_TARGET = "pCO2"）
  4) pCO2_prior_used + r_star_pred （旧版兼容）

输出（单文件）:
  /data/wang/Result_pCO2/Datasets/GLOBAL_pCO2_monthly_YYYYMM_YYYYMM_1deg_v1.nc
  （文件名根据实际年月范围自动生成）

Pipeline:
  1) 解析/重建 pCO2_pred
  2) 点 -> 1°格点（月平均；nearest-cell bin）
  3) Mallow mask = offshore_mask only
  4) 可选 Gaussian smoothing（preserve NaN; no domain expansion）
  5) 不做插值填补
  6) 对有效值施加 floor(min)
  7) 若启用 BIOME_NAN_MASK，则 Basins_provinces.nc 中 MeanBiomes 对应格点为 NaN 的位置最终强制为 NaN
  8) 写 NetCDF（single file; time, latitude, longitude）

Notes:
  - offshore_mask 源网格通常为 0.25°（lat, lon in 0..360），映射到 1° 网格采用最近邻索引。
  - CSV 经度可为 [-180,180] 或其他形式；输出 lon 为 [-179.5..179.5]，mask 映射内部使用 lon%360。
  - 默认仅保留 1982 年及以后数据。
  - 可通过开关决定是否仅保留 Zone 非 NaN 的记录。
  - 可通过开关决定是否启用基于 Basins_provinces.nc / MeanBiomes 的 NaN 后处理。
"""

import os
import argparse
import numpy as np
import pandas as pd
from netCDF4 import Dataset
from datetime import datetime, timezone
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp


# ===================== Defaults =====================
OUT_DIR_DEFAULT = "/data/wang/Result_pCO2/Datasets"

MASK_NC_DEFAULT = "/data/wang/Mask_File/land_mask_0m.nc"   # expects offshore_mask
BIOMES_NC_DEFAULT = "/data/wang/Basins_provinces.nc"
BIOMES_VAR_DEFAULT = "MeanBiomes"

START_YEAR_DEFAULT = 1982
END_YEAR_DEFAULT = 2024

# smoothing
PREFILTER_ENABLE_DEFAULT = True
GAUSS_RADIUS_KM_DEFAULT = 100.0
MIN_SUPPORT_FRAC_DEFAULT = 0.5

# floor
MIN_VALUE_FLOOR_DEFAULT = 0.001

# valid range
PCO2_VALID_MIN_DEFAULT = 0.001
PCO2_VALID_MAX_DEFAULT = 2000.0

# Zone filtering switches
FILTER_BY_ZONE_VALID_DEFAULT = False
WRITE_ZONE_CLEAN_TO_OUTPUT_DEFAULT = True

# biome NaN post-mask switch
# auto: if Modeling_mode_used is zonewise -> on; if global -> off
BIOME_NAN_MASK_MODE_DEFAULT = "on"   # {"on", "off"}
if BIOME_NAN_MASK_MODE_DEFAULT == "on":
    PRED_CSV_DEFAULT = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_pred.csv"
elif BIOME_NAN_MASK_MODE_DEFAULT == "off":
    PRED_CSV_DEFAULT = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_predglobal.csv"
else:
    PRED_CSV_DEFAULT = "/data/wang/Result_pCO2/allnopco2/SOCAT_TRAIN_with_pred.csv"

# metadata
TITLE = "Global monthly surface pCO2 (ML reconstruction) 1deg"
INSTITUTION = "Fudan University"
CREATOR_NAME = "Wang et al."
LICENSE = "CC BY 4.0"
CONVENTIONS = "CF-1.8, ACDD-1.3"
REF_DATE = "1950-01-01T00:00:00Z"

# current prediction columns
PCO2_PRED_COL = "pCO2_pred"
PCO2_PRED_STD_COL = "pCO2_pred_std"
TARGET_PRED_COL = "target_pred"
FINAL_TARGET_USED_COL = "Final_target_used"
MODELING_MODE_USED_COL = "Modeling_mode_used"

# current r_star_T reconstruction candidates
PRIOR_CLIM_CANDIDATES = [
    "pCO2_prior_clim_mapped", "pco2_prior_clim_mapped",
    "pCO2_prior_clim", "pco2_prior_clim",
    "pCO2_prior_clim_used"
]
DELTA_T_CANDIDATES = ["delta_pCO2_T", "delta_pCO2_T_used"]

# legacy fallback
LEGACY_PRIOR_COL = "pCO2_prior_used"
LEGACY_RSTAR_PRED_COL = "r_star_pred"


# ===================== 1° grid definition =====================
def build_1deg_grid():
    lats = np.arange(-89.5, 90.0, 1.0, dtype=np.float32)    # 180
    lons = np.arange(-179.5, 180.0, 1.0, dtype=np.float32)  # 360
    return lats, lons


# ===================== Basic utils =====================
def normalize_lon_to_m180_180(lon_arr: np.ndarray) -> np.ndarray:
    return (lon_arr + 180.0) % 360.0 - 180.0


def lon_to_0_360(lon_deg):
    x = np.asarray(lon_deg, np.float64)
    out = np.mod(x, 360.0)
    out[out < 0] += 360.0
    return out


def days_since_ref(dt_utc: datetime, ref=REF_DATE) -> float:
    ref_dt = datetime.fromisoformat(ref.replace("Z", "+00:00"))
    return (dt_utc - ref_dt).total_seconds() / 86400.0


def mid_month_dt(y: int, m: int) -> datetime:
    return datetime(int(y), int(m), 15, tzinfo=timezone.utc)


def nearest_index_nonuniform(axis: np.ndarray, values: np.ndarray) -> np.ndarray:
    axis = np.asarray(axis, np.float64)
    values = np.asarray(values, np.float64)
    idx = np.searchsorted(axis, values, side="left")
    idx0 = np.clip(idx - 1, 0, axis.size - 1)
    idx1 = np.clip(idx, 0, axis.size - 1)
    choose_right = np.abs(axis[idx1] - values) < np.abs(axis[idx0] - values)
    return np.where(choose_right, idx1, idx0).astype(np.int64)


def find_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def bincount_mean(i: np.ndarray, j: np.ndarray, val: np.ndarray, ny: int, nx: int):
    flat = i * nx + j
    m = np.isfinite(val)
    CNT = np.bincount(flat[m], minlength=ny * nx)
    if not np.any(m):
        return np.full((ny, nx), np.nan, dtype=np.float32), CNT.reshape(ny, nx).astype(np.int32)

    SUM = np.bincount(flat[m], weights=val[m].astype(np.float64), minlength=ny * nx)
    SUM = SUM.reshape(ny, nx)
    CNT = CNT.reshape(ny, nx)
    with np.errstate(invalid="ignore", divide="ignore"):
        MEAN = SUM / np.where(CNT == 0, 1, CNT)
    MEAN[CNT == 0] = np.nan
    return MEAN.astype(np.float32), CNT.astype(np.int32)


def apply_floor(A: np.ndarray, floor_val: float) -> np.ndarray:
    X = A.copy()
    m = np.isfinite(X)
    if np.any(m):
        X[m] = np.maximum(X[m], float(floor_val))
    return X


def clean_zone_column(df: pd.DataFrame) -> pd.DataFrame:
    if "Zone" not in df.columns:
        return df
    df = df.copy()
    df["Zone"] = pd.to_numeric(df["Zone"], errors="coerce")
    return df


def infer_modeling_mode_from_csv(df: pd.DataFrame) -> str:
    if MODELING_MODE_USED_COL in df.columns:
        s = df[MODELING_MODE_USED_COL].dropna().astype(str).str.strip().str.lower()
        if not s.empty:
            vals = set(s.unique().tolist())
            if "zonewise" in vals:
                return "zonewise"
            if "global" in vals:
                return "global"

    if "Zone_used_for_model" in df.columns:
        s = df["Zone_used_for_model"].dropna().astype(str).str.strip()
        if not s.empty:
            vals = set(s.unique().tolist())
            if vals == {"GLOBAL"}:
                return "global"
            return "zonewise"

    return "unknown"


def resolve_biome_nan_mask_enabled(mask_mode: str, inferred_modeling_mode: str) -> bool:
    mask_mode = str(mask_mode).strip().lower()
    if mask_mode not in {"auto", "on", "off"}:
        raise ValueError("biome_nan_mask_mode must be one of {'auto','on','off'}")

    if mask_mode == "on":
        return True
    if mask_mode == "off":
        return False

    if inferred_modeling_mode == "zonewise":
        return True
    return False


# ===================== Prediction reconstruction =====================
def _all_tokens_are_pco2(series_like) -> bool:
    s = pd.Series(series_like).dropna().astype(str).str.strip()
    if s.empty:
        return False
    tokens = set()
    for item in s:
        for t in item.split(","):
            tt = t.strip()
            if tt:
                tokens.add(tt)
    return len(tokens) > 0 and tokens == {"pCO2"}


def infer_prediction_source(df: pd.DataFrame):
    if PCO2_PRED_COL in df.columns:
        x = pd.to_numeric(df[PCO2_PRED_COL], errors="coerce")
        if x.notna().any():
            return x, "direct_current_column:pCO2_pred"

    target_col = TARGET_PRED_COL if TARGET_PRED_COL in df.columns else None
    prior_clim_col = find_first_existing(df.columns, PRIOR_CLIM_CANDIDATES)
    delta_t_col = find_first_existing(df.columns, DELTA_T_CANDIDATES)

    if (target_col is not None) and (prior_clim_col is not None) and (delta_t_col is not None):
        target = pd.to_numeric(df[target_col], errors="coerce")
        prior_clim = pd.to_numeric(df[prior_clim_col], errors="coerce")
        delta_t = pd.to_numeric(df[delta_t_col], errors="coerce")
        x = target + prior_clim + delta_t
        if x.notna().any():
            scheme = f"reconstructed_current_r_star_T:{target_col}+{prior_clim_col}+{delta_t_col}"
            return x, scheme

    if (target_col is not None) and (FINAL_TARGET_USED_COL in df.columns):
        if _all_tokens_are_pco2(df[FINAL_TARGET_USED_COL]):
            x = pd.to_numeric(df[target_col], errors="coerce")
            if x.notna().any():
                return x, "reconstructed_current_direct_target:target_pred"

    if (LEGACY_PRIOR_COL in df.columns) and (LEGACY_RSTAR_PRED_COL in df.columns):
        prior = pd.to_numeric(df[LEGACY_PRIOR_COL], errors="coerce")
        rstar = pd.to_numeric(df[LEGACY_RSTAR_PRED_COL], errors="coerce")
        x = prior + rstar
        if x.notna().any():
            return x, "legacy_reconstruction:pCO2_prior_used+r_star_pred"

    raise ValueError(
        "无法从输入 CSV 中解析 pCO2 预测值。需要满足以下任一条件：\n"
        "1) 存在且可用列: pCO2_pred\n"
        "2) 存在且可用列: target_pred + pCO2_prior_clim_* + delta_pCO2_T\n"
        "3) 存在且可用列: target_pred，并且 Final_target_used == pCO2\n"
        "4) 旧版兼容列: pCO2_prior_used + r_star_pred"
    )


# ===================== Gaussian smoothing =====================
def _gaussian_kernel1d(sigma_gp, truncate=3.0):
    sigma = float(max(1e-6, sigma_gp))
    half = int(max(1, np.ceil(truncate * sigma)))
    x = np.arange(-half, half + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    k /= k.sum() + 1e-12
    return k


def _km_to_grid_sigma(radius_km: float, ddeg: float):
    if ddeg == 0:
        return 1.0
    return max(0.5, (radius_km / 111.0) / abs(ddeg))


def _reflect_indices(idx, N):
    idx = idx.copy()
    idx[idx < 0] = -idx[idx < 0] - 1
    idx[idx >= N] = 2 * N - idx[idx >= N] - 1
    return np.clip(idx, 0, N - 1)


def _norm_conv_line_wrap(x, mask01, kernel):
    acc = np.zeros_like(x, float)
    wsum = np.zeros_like(x, float)
    N = x.shape[0]
    m = kernel.size // 2
    for r, w in enumerate(kernel):
        shift = r - m
        xs = np.roll(x, shift)
        ms = np.roll(mask01, shift)
        xs = np.where(ms > 0, xs, 0.0)
        acc += w * xs
        wsum += w * ms
    out = np.where(wsum > 0, acc / np.maximum(wsum, 1e-12), np.nan)
    return out, wsum


def _norm_conv_line_reflect(x, mask01, kernel):
    acc = np.zeros_like(x, float)
    wsum = np.zeros_like(x, float)
    N = x.shape[0]
    m = kernel.size // 2
    base = np.arange(N, dtype=np.int64)
    for r, w in enumerate(kernel):
        shift = r - m
        idx = _reflect_indices(base + shift, N)
        xs = x[idx]
        ms = mask01[idx]
        xs = np.where(ms > 0, xs, 0.0)
        acc += w * xs
        wsum += w * ms
    out = np.where(wsum > 0, acc / np.maximum(wsum, 1e-12), np.nan)
    return out, wsum


def gaussian_smooth_preserve(V: np.ndarray, lats: np.ndarray, lons: np.ndarray,
                             radius_km: float, min_support_frac: float = 0.5) -> np.ndarray:
    V = V.astype(np.float64)
    Ny, Nx = V.shape
    mask0 = np.isfinite(V).astype(np.float64)
    out = V.copy()

    dlon = float(np.nanmedian(np.diff(lons))) if Nx > 1 else 1.0
    for i in range(Ny):
        cosphi = max(0.1, abs(np.cos(np.deg2rad(float(lats[i])))))
        sigma_lon = _km_to_grid_sigma(radius_km / max(cosphi, 1e-6), dlon)
        k_lon = _gaussian_kernel1d(sigma_lon, 3.0)
        sm, ws = _norm_conv_line_wrap(out[i, :], mask0[i, :], k_lon)
        good = ws >= (min_support_frac * k_lon.sum())
        sm = np.where((mask0[i, :] > 0) & (~good), out[i, :], sm)
        sm = np.where((mask0[i, :] > 0) & np.isnan(sm), out[i, :], sm)
        sm = np.where(mask0[i, :] > 0, sm, np.nan)
        out[i, :] = sm

    mask1 = np.isfinite(out).astype(np.float64)
    dlat = float(np.nanmedian(np.diff(lats))) if Ny > 1 else 1.0
    sigma_lat = _km_to_grid_sigma(radius_km, dlat)
    k_lat = _gaussian_kernel1d(sigma_lat, 3.0)
    final = out.copy()
    for j in range(Nx):
        sm, ws = _norm_conv_line_reflect(out[:, j], mask1[:, j], k_lat)
        good = ws >= (min_support_frac * k_lat.sum())
        sm = np.where((mask0[:, j] > 0) & (~good), out[:, j], sm)
        sm = np.where((mask0[:, j] > 0) & np.isnan(sm), out[:, j], sm)
        sm = np.where(mask0[:, j] > 0, sm, np.nan)
        final[:, j] = sm

    final = np.where(mask0 > 0, final, np.nan)
    return final.astype(np.float32)


# ===================== Shared axis/index helpers =====================
def _as_sorted_axis_and_order(axis):
    axis = np.asarray(axis, np.float64)
    order = np.argsort(axis)
    sorted_axis = axis[order]
    return sorted_axis, order


def nearest_index_on_sorted_axis(sorted_axis, values):
    v = np.asarray(values, np.float64)
    idx = np.searchsorted(sorted_axis, v, side="left")
    idx0 = np.clip(idx - 1, 0, sorted_axis.size - 1)
    idx1 = np.clip(idx, 0, sorted_axis.size - 1)
    choose_right = np.abs(sorted_axis[idx1] - v) < np.abs(sorted_axis[idx0] - v)
    return np.where(choose_right, idx1, idx0).astype(np.int64)


# ===================== Offshore mask mapping =====================
_MASK_CACHE = {
    "path": None,
    "lat": None,
    "lon": None,
    "mask2d": None,
    "lat_sorted": None,
    "lat_order": None,
    "lon_sorted": None,
    "lon_order": None,
}


def load_offshore_mask(mask_nc_path):
    global _MASK_CACHE
    if _MASK_CACHE["path"] == mask_nc_path:
        return

    with Dataset(mask_nc_path, "r") as nc:
        lat = nc["lat"][:].astype(np.float64)
        lon = nc["lon"][:].astype(np.float64)
        var = nc["offshore_mask"]
        m = var[:].astype(np.uint8)

        dims = tuple(var.dimensions)
        if dims == ("lat", "lon"):
            mask2d = m
        elif dims == ("lon", "lat"):
            mask2d = m.T
        else:
            raise ValueError(
                f"offshore_mask dimensions not supported: {dims}. "
                "Expected ('lat','lon') or ('lon','lat')."
            )

    lat_sorted, lat_order = _as_sorted_axis_and_order(lat)
    lon_sorted, lon_order = _as_sorted_axis_and_order(lon)

    _MASK_CACHE = {
        "path": mask_nc_path,
        "lat": lat,
        "lon": lon,
        "mask2d": mask2d,   # (lat, lon)
        "lat_sorted": lat_sorted,
        "lat_order": lat_order,
        "lon_sorted": lon_sorted,
        "lon_order": lon_order,
    }


def map_offshore_mask_to_1deg(mask_nc_path, target_lats_1deg, target_lons_1deg):
    load_offshore_mask(mask_nc_path)

    lat_sorted = _MASK_CACHE["lat_sorted"]
    lon_sorted = _MASK_CACHE["lon_sorted"]
    lat_order = _MASK_CACHE["lat_order"]
    lon_order = _MASK_CACHE["lon_order"]
    M2 = _MASK_CACHE["mask2d"]

    tgt_lon_0360 = lon_to_0_360(target_lons_1deg)

    idx_lat_sorted = nearest_index_on_sorted_axis(lat_sorted, target_lats_1deg)
    idx_lon_sorted = nearest_index_on_sorted_axis(lon_sorted, tgt_lon_0360)

    lat_idx = lat_order[idx_lat_sorted]
    lon_idx = lon_order[idx_lon_sorted]

    mapped = M2[np.ix_(lat_idx, lon_idx)].astype(np.uint8)
    return mapped


# ===================== Biome NaN mask mapping =====================
_BIOME_CACHE = {
    "path": None,
    "var_name": None,
    "lat": None,
    "lon": None,
    "nanmask2d": None,
    "lat_sorted": None,
    "lat_order": None,
    "lon_sorted": None,
    "lon_order": None,
}


def load_biome_nan_mask(biomes_nc_path, biome_var):
    global _BIOME_CACHE
    if _BIOME_CACHE["path"] == biomes_nc_path and _BIOME_CACHE["var_name"] == biome_var:
        return

    with Dataset(biomes_nc_path, "r") as nc:
        if "lat" not in nc.variables or "lon" not in nc.variables:
            raise KeyError(f"Basins_provinces.nc missing lat/lon: {biomes_nc_path}")
        if biome_var not in nc.variables:
            raise KeyError(f"Variable '{biome_var}' not found in {biomes_nc_path}")

        lat = nc["lat"][:].astype(np.float64)
        lon = nc["lon"][:].astype(np.float64)

        var = nc[biome_var]
        arr = var[:]

        if np.ma.isMaskedArray(arr):
            data = np.asarray(arr.filled(np.nan), dtype=np.float64)
        else:
            data = np.asarray(arr, dtype=np.float64)

        if data.ndim != 2:
            raise ValueError(f"Variable '{biome_var}' must be 2D, got shape={data.shape}")

        dims = tuple(var.dimensions)
        if dims == ("lat", "lon"):
            data_latlon = data
        elif dims == ("lon", "lat"):
            data_latlon = data.T
        else:
            raise ValueError(
                f"Variable '{biome_var}' dimensions not supported: {dims}. "
                "Expected ('lat','lon') or ('lon','lat')."
            )

        expected_shape = (len(lat), len(lon))
        if data_latlon.shape != expected_shape:
            raise ValueError(
                f"After dimension normalization, {biome_var} shape = {data_latlon.shape}, "
                f"expected {expected_shape} = (len(lat), len(lon))"
            )

        nanmask2d = np.isnan(data_latlon)

    lat_sorted, lat_order = _as_sorted_axis_and_order(lat)
    lon_sorted, lon_order = _as_sorted_axis_and_order(lon)

    _BIOME_CACHE = {
        "path": biomes_nc_path,
        "var_name": biome_var,
        "lat": lat,
        "lon": lon,
        "nanmask2d": nanmask2d.astype(bool),  # (lat, lon)
        "lat_sorted": lat_sorted,
        "lat_order": lat_order,
        "lon_sorted": lon_sorted,
        "lon_order": lon_order,
    }


def map_biome_nan_mask_to_1deg(biomes_nc_path, biome_var, target_lats_1deg, target_lons_1deg):
    load_biome_nan_mask(biomes_nc_path, biome_var)

    lat_sorted = _BIOME_CACHE["lat_sorted"]
    lon_sorted = _BIOME_CACHE["lon_sorted"]
    lat_order = _BIOME_CACHE["lat_order"]
    lon_order = _BIOME_CACHE["lon_order"]
    M2 = _BIOME_CACHE["nanmask2d"]   # (lat, lon)
    lon_nc = _BIOME_CACHE["lon"]

    if np.nanmax(lon_nc) > 180.0:
        tgt_lon_query = lon_to_0_360(target_lons_1deg)
    else:
        tgt_lon_query = normalize_lon_to_m180_180(target_lons_1deg)

    idx_lat_sorted = nearest_index_on_sorted_axis(lat_sorted, target_lats_1deg)
    idx_lon_sorted = nearest_index_on_sorted_axis(lon_sorted, tgt_lon_query)

    lat_idx = lat_order[idx_lat_sorted]
    lon_idx = lon_order[idx_lon_sorted]

    mapped = M2[np.ix_(lat_idx, lon_idx)].astype(bool)
    return mapped


# ===================== NetCDF single-file output =====================
def build_output_filename(out_dir, months):
    start_y, start_m = months[0]
    end_y, end_m = months[-1]
    if (start_y, start_m) == (end_y, end_m):
        name = f"GLOBAL_pCO2_monthly_{start_y:04d}{start_m:02d}_1deg_v1.nc"
    else:
        name = f"GLOBAL_pCO2_monthly_{start_y:04d}{start_m:02d}_{end_y:04d}{end_m:02d}_1deg_v1.nc"
    return os.path.join(out_dir, name)


def ensure_output_file(path_nc: str, lats: np.ndarray, lons: np.ndarray, months,
                       prediction_scheme: str, prediction_source_csv: str,
                       filter_by_zone_valid: bool,
                       biome_nan_mask_enabled: bool,
                       biomes_nc: str,
                       biome_var: str):
    os.makedirs(os.path.dirname(path_nc), exist_ok=True)
    if os.path.exists(path_nc):
        os.remove(path_nc)

    ds = Dataset(path_nc, "w", format="NETCDF4")

    Ny, Nx = len(lats), len(lons)
    Nt = len(months)

    ds.createDimension("time", Nt)
    ds.createDimension("latitude", Ny)
    ds.createDimension("longitude", Nx)

    vtime = ds.createVariable("time", "f4", ("time",))
    vlat = ds.createVariable("latitude", "f4", ("latitude",))
    vlon = ds.createVariable("longitude", "f4", ("longitude",))

    vlat[:] = lats.astype(np.float32)
    vlon[:] = lons.astype(np.float32)
    vtime[:] = np.array([days_since_ref(mid_month_dt(y, m)) for (y, m) in months], dtype=np.float32)

    vtime.standard_name = "time"
    vtime.long_name = "time"
    vtime.units = f"days since {REF_DATE}"
    vtime.calendar = "gregorian"
    vtime.axis = "T"

    vlat.standard_name = "latitude"
    vlat.units = "degree_north"
    vlat.axis = "Y"

    vlon.standard_name = "longitude"
    vlon.units = "degree_east"
    vlon.axis = "X"

    chunks_lat = min(180, Ny if Ny > 0 else 1)
    chunks_lon = min(180, Nx if Nx > 0 else 1)

    vp = ds.createVariable(
        "pCO2", "f4", ("time", "latitude", "longitude"),
        zlib=True, complevel=4, shuffle=True,
        chunksizes=(1, max(1, chunks_lat), max(1, chunks_lon)),
        fill_value=np.float32(np.nan)
    )
    vp.long_name = "Surface partial pressure of CO2"
    vp.standard_name = "surface_partial_pressure_of_carbon_dioxide_in_sea_water"
    vp.units = "uatm"

    start_y, start_m = months[0]
    end_y, end_m = months[-1]

    ds.Conventions = CONVENTIONS
    ds.title = TITLE
    ds.institution = INSTITUTION
    ds.creator_name = CREATOR_NAME
    ds.license = LICENSE
    ds.history = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + " : Creation"
    ds.id = f"GLOBAL_pCO2_monthly_{start_y:04d}{start_m:02d}_{end_y:04d}{end_m:02d}_1deg_v1"
    ds.dataset_id = ds.id
    ds.cdm_data_type = "Grid"
    ds.featureType = "grid"
    ds.time_coverage_start = f"{start_y:04d}-{start_m:02d}-15T00:00:00Z"
    ds.time_coverage_end = f"{end_y:04d}-{end_m:02d}-15T00:00:00Z"
    ds.geospatial_lat_min = float(np.nanmin(lats))
    ds.geospatial_lat_max = float(np.nanmax(lats))
    ds.geospatial_lon_min = float(np.nanmin(lons))
    ds.geospatial_lon_max = float(np.nanmax(lons))
    ds.geospatial_lat_resolution = 1.0
    ds.geospatial_lon_resolution = 1.0
    ds.comment = (
        "Monthly mean surface pCO2; points->grid mean; offshore_mask only; "
        "optional smooth; no fill; all months in one file; "
        "optional biome-NaN post-mask for zonewise outputs."
    )
    ds.prediction_source_csv = prediction_source_csv
    ds.prediction_scheme = prediction_scheme
    ds.filter_by_zone_valid = int(bool(filter_by_zone_valid))
    ds.biome_nan_mask_enabled = int(bool(biome_nan_mask_enabled))
    ds.biomes_nc = str(biomes_nc if biome_nan_mask_enabled else "NOT_USED")
    ds.biome_var = str(biome_var if biome_nan_mask_enabled else "NOT_USED")

    ds.close()


def write_time_slice(path_nc, time_index, field2d):
    with Dataset(path_nc, "r+") as ds:
        vp = ds["pCO2"]
        data = np.ma.masked_invalid(np.array(field2d, dtype=np.float32))
        vp[time_index:time_index + 1, :, :] = data[np.newaxis, :, :]


# ===================== Month compute task =====================
def _process_one_month(task):
    yy = int(task["yy"])
    mm = int(task["mm"])
    floor = float(task["floor"])

    target_lats = task["grid_lats"]
    target_lons = task["grid_lons"]
    Ny, Nx = len(target_lats), len(target_lons)

    plat = task["lat"].astype(np.float64)
    plon = normalize_lon_to_m180_180(task["lon"].astype(np.float64))
    pval = task["pco2"].astype(np.float64)

    ii = nearest_index_nonuniform(target_lats, plat)
    jj = nearest_index_nonuniform(target_lons, plon)
    V_raw, CNT = bincount_mean(ii, jj, pval, Ny, Nx)
    have = CNT > 0

    offshore_1deg = task["offshore_1deg"]  # 1 land/nearshore; 0 ocean
    Mallow = (offshore_1deg == 0)

    V = np.where(Mallow & have, V_raw, np.nan)
    V = apply_floor(V, floor)

    if task["prefilter_enable"]:
        V = gaussian_smooth_preserve(
            V, target_lats, target_lons,
            radius_km=float(task["gauss_radius_km"]),
            min_support_frac=float(task["min_support_frac"])
        )
        V = apply_floor(V, floor)

    V_out = np.where(Mallow, V, np.nan).astype(np.float32)
    V_out = apply_floor(V_out, floor)

    if bool(task["biome_nan_mask_enabled"]):
        biome_nan_1deg = task["biome_nan_1deg"]
        V_out = np.where(biome_nan_1deg, np.nan, V_out).astype(np.float32)

    vmin = float(np.nanmin(V_out)) if np.isfinite(V_out).any() else np.nan
    vmax = float(np.nanmax(V_out)) if np.isfinite(V_out).any() else np.nan
    valid = int(np.isfinite(V_out).sum())

    return {
        "yy": yy,
        "mm": mm,
        "field": V_out,
        "vmin": vmin,
        "vmax": vmax,
        "valid": valid
    }


# ===================== Main =====================
def main():
    parser = argparse.ArgumentParser(allow_abbrev=False)
    parser.add_argument("--pred_csv", default=PRED_CSV_DEFAULT)
    parser.add_argument("--out_dir", default=OUT_DIR_DEFAULT)
    parser.add_argument("--mask_nc", default=MASK_NC_DEFAULT)
    parser.add_argument("--biomes_nc", default=BIOMES_NC_DEFAULT)
    parser.add_argument("--biome_var", default=BIOMES_VAR_DEFAULT)

    parser.add_argument("--start_year", type=int, default=START_YEAR_DEFAULT)
    parser.add_argument("--end_year", type=int, default=END_YEAR_DEFAULT)

    parser.add_argument("--prefilter_enable", type=int, default=1 if PREFILTER_ENABLE_DEFAULT else 0)
    parser.add_argument("--gauss_radius_km", type=float, default=GAUSS_RADIUS_KM_DEFAULT)
    parser.add_argument("--min_support_frac", type=float, default=MIN_SUPPORT_FRAC_DEFAULT)

    parser.add_argument("--floor", type=float, default=MIN_VALUE_FLOOR_DEFAULT)
    parser.add_argument("--pco2_valid_min", type=float, default=PCO2_VALID_MIN_DEFAULT)
    parser.add_argument("--pco2_valid_max", type=float, default=PCO2_VALID_MAX_DEFAULT)
    parser.add_argument("--workers", type=int, default=min(24, os.cpu_count() or 1))

    parser.add_argument(
        "--filter_by_zone_valid",
        type=int,
        default=1 if FILTER_BY_ZONE_VALID_DEFAULT else 0,
        help="1: only keep rows with non-NaN Zone; 0: do not filter by Zone"
    )
    parser.add_argument(
        "--write_zone_clean_to_output",
        type=int,
        default=1 if WRITE_ZONE_CLEAN_TO_OUTPUT_DEFAULT else 0,
        help="1: clean Zone column internally if present; 0: ignore Zone cleaning"
    )
    parser.add_argument(
        "--biome_nan_mask_mode",
        default=BIOME_NAN_MASK_MODE_DEFAULT,
        choices=["auto", "on", "off"],
        help="auto: zonewise on / global off; on: always apply; off: never apply"
    )

    args, _ = parser.parse_known_args()

    pred_csv = str(args.pred_csv)
    out_dir = str(args.out_dir)
    mask_nc = str(args.mask_nc)
    biomes_nc = str(args.biomes_nc)
    biome_var = str(args.biome_var)
    os.makedirs(out_dir, exist_ok=True)

    filter_by_zone_valid = bool(int(args.filter_by_zone_valid))
    write_zone_clean_to_output = bool(int(args.write_zone_clean_to_output))

    grid_lats, grid_lons = build_1deg_grid()

    offshore_1deg = map_offshore_mask_to_1deg(
        mask_nc,
        grid_lats.astype(np.float64),
        grid_lons.astype(np.float64)
    )
    if offshore_1deg.shape != (len(grid_lats), len(grid_lons)):
        raise RuntimeError(
            f"offshore_1deg shape mismatch: {offshore_1deg.shape} != {(len(grid_lats), len(grid_lons))}"
        )

    df = pd.read_csv(pred_csv, low_memory=False)

    required_base_cols = ["Year", "Month", "Latitude", "Longitude"]
    missing_base = [c for c in required_base_cols if c not in df.columns]
    if missing_base:
        raise ValueError(f"Missing required columns in prediction CSV: {missing_base}")

    inferred_modeling_mode = infer_modeling_mode_from_csv(df)
    biome_nan_mask_enabled = resolve_biome_nan_mask_enabled(
        mask_mode=args.biome_nan_mask_mode,
        inferred_modeling_mode=inferred_modeling_mode
    )

    if write_zone_clean_to_output and ("Zone" in df.columns):
        df = clean_zone_column(df)

    pco2_series, prediction_scheme = infer_prediction_source(df)
    df[PCO2_PRED_COL] = pco2_series

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df["Month"] = pd.to_numeric(df["Month"], errors="coerce")
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
    df[PCO2_PRED_COL] = pd.to_numeric(df[PCO2_PRED_COL], errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["Year", "Month", "Latitude", "Longitude", PCO2_PRED_COL]).copy()

    df["Year"] = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)

    df = df[(df["Month"] >= 1) & (df["Month"] <= 12)].copy()
    df["Longitude"] = normalize_lon_to_m180_180(df["Longitude"].astype(float).values)

    df = df[
        (df["Year"] >= int(args.start_year)) &
        (df["Year"] <= int(args.end_year))
    ].copy()

    if filter_by_zone_valid:
        if "Zone" not in df.columns:
            raise ValueError("filter_by_zone_valid=True，但输入 CSV 中不存在 Zone 字段。")
        df = df[df["Zone"].notna()].copy()

    df = df[
        (df[PCO2_PRED_COL] >= float(args.pco2_valid_min)) &
        (df[PCO2_PRED_COL] <= float(args.pco2_valid_max))
    ].copy()

    if df.empty:
        print("[DONE] No valid rows remain after filtering.")
        print(f"[INFO] filter_by_zone_valid = {filter_by_zone_valid}")
        print(f"[INFO] inferred_modeling_mode = {inferred_modeling_mode}")
        print(f"[INFO] biome_nan_mask_enabled = {biome_nan_mask_enabled}")
        return

    biome_nan_1deg = np.zeros((len(grid_lats), len(grid_lons)), dtype=bool)
    if biome_nan_mask_enabled:
        if not os.path.exists(biomes_nc):
            raise FileNotFoundError(
                f"biome_nan_mask_enabled=True, but BIOMES_NC not found: {biomes_nc}"
            )

        biome_nan_1deg = map_biome_nan_mask_to_1deg(
            biomes_nc,
            biome_var,
            grid_lats.astype(np.float64),
            grid_lons.astype(np.float64)
        )

    months = sorted(set(zip(df["Year"].tolist(), df["Month"].tolist())))
    if not months:
        print("[DONE] No valid months in input after year/month filtering.")
        print(f"[INFO] filter_by_zone_valid = {filter_by_zone_valid}")
        print(f"[INFO] inferred_modeling_mode = {inferred_modeling_mode}")
        print(f"[INFO] biome_nan_mask_enabled = {biome_nan_mask_enabled}")
        return

    out_nc = build_output_filename(out_dir, months)
    ensure_output_file(
        out_nc,
        grid_lats,
        grid_lons,
        months,
        prediction_scheme=prediction_scheme,
        prediction_source_csv=pred_csv,
        filter_by_zone_valid=filter_by_zone_valid,
        biome_nan_mask_enabled=biome_nan_mask_enabled,
        biomes_nc=biomes_nc,
        biome_var=biome_var
    )

    time_index_map = {(y, m): k for k, (y, m) in enumerate(months)}
    groups = {(y, m): g for (y, m), g in df.groupby(["Year", "Month"], sort=False)}

    tasks = []
    for (yy, mm) in months:
        g = groups.get((yy, mm), None)
        if g is None or g.empty:
            continue
        tasks.append({
            "yy": yy,
            "mm": mm,
            "lat": g["Latitude"].to_numpy(np.float64, copy=False),
            "lon": g["Longitude"].to_numpy(np.float64, copy=False),
            "pco2": g[PCO2_PRED_COL].to_numpy(np.float64, copy=False),
            "grid_lats": grid_lats,
            "grid_lons": grid_lons,
            "offshore_1deg": offshore_1deg,
            "prefilter_enable": bool(int(args.prefilter_enable)),
            "gauss_radius_km": float(args.gauss_radius_km),
            "min_support_frac": float(args.min_support_frac),
            "floor": float(args.floor),
            "biome_nan_mask_enabled": biome_nan_mask_enabled,
            "biome_nan_1deg": biome_nan_1deg,
        })

    workers = max(1, int(args.workers))
    mp_context = mp.get_context("fork") if hasattr(mp, "get_context") else None
    executor_kwargs = {"max_workers": workers}
    if mp_context is not None:
        executor_kwargs["mp_context"] = mp_context

    total = 0
    stats_lines = []

    with ProcessPoolExecutor(**executor_kwargs) as ex:
        futs = {ex.submit(_process_one_month, t): (t["yy"], t["mm"]) for t in tasks}
        for fut in as_completed(futs):
            yy_, mm_ = futs[fut]
            try:
                info = fut.result()
                tidx = time_index_map[(info["yy"], info["mm"])]
                write_time_slice(out_nc, tidx, info["field"])
                total += 1

                line = (
                    f"{info['yy']}-{info['mm']:02d} "
                    f"min={info['vmin']:.3f} max={info['vmax']:.3f} valid={info['valid']}"
                )
                stats_lines.append(line)

                print(
                    f"[OK] {info['yy']}-{info['mm']:02d} -> time_index={tidx} "
                    f"min={info['vmin']:.3f} max={info['vmax']:.3f} valid={info['valid']}"
                )
            except Exception as e:
                print(f"[ERR] {yy_}-{mm_:02d}: {e}")

    with Dataset(out_nc, "r+") as ds:
        ds.setncattr("n_time_written", int(total))
        ds.setncattr("last_write_history", datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"))
        ds.setncattr("prediction_source_csv", pred_csv)
        ds.setncattr("prediction_scheme", prediction_scheme)
        ds.setncattr("filter_by_zone_valid", int(filter_by_zone_valid))
        ds.setncattr("inferred_modeling_mode", inferred_modeling_mode)
        ds.setncattr("biome_nan_mask_enabled", int(biome_nan_mask_enabled))
        ds.setncattr("biomes_nc", biomes_nc if biome_nan_mask_enabled else "NOT_USED")
        ds.setncattr("biome_var", biome_var if biome_nan_mask_enabled else "NOT_USED")
        if stats_lines:
            ds.setncattr("last_write_stats", " | ".join(stats_lines[:200]))

    print(f"[DONE] Written/updated {total} monthly slices into {out_nc}")
    print(f"[INFO] prediction_scheme = {prediction_scheme}")
    print(f"[INFO] filter_by_zone_valid = {filter_by_zone_valid}")
    print(f"[INFO] write_zone_clean_to_output = {write_zone_clean_to_output}")
    print(f"[INFO] inferred_modeling_mode = {inferred_modeling_mode}")
    print(f"[INFO] biome_nan_mask_enabled = {biome_nan_mask_enabled}")
    print(f"[INFO] biomes_nc = {biomes_nc if biome_nan_mask_enabled else 'NOT_USED'}")
    print(f"[INFO] biome_var = {biome_var if biome_nan_mask_enabled else 'NOT_USED'}")


if __name__ == "__main__":
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
    main()